# T&D — Priorização de Melhorias Produto Físico

Notebook produtivo para extrair, processar, validar e publicar o Farol de
Priorização de Melhorias de Produto Físico/T&D.

Este notebook consolida:
- SQL executivo de extração no BigQuery;
- funções analíticas e regras de negócio;
- preparação, score e classificação de priorização;
- QA dos dados e dos payloads finais;
- atualização da planilha oficial existente.

## Estrutura

| Bloco | Descrição |
|---|---|
| 1 | Configuração do ambiente |
| 2 | Parâmetros do pipeline |
| 3 | SQL executivo e scorecard |
| 4 | Pós-extração dos DataFrames fonte |
| 5 | Funções utilitárias e regras de negócio |
| 6 | Preparação e normalização dos DataFrames |
| 7 | Classificação e score de priorização |
| 8 | Diagnósticos executivos |
| 9 | QA e validações |
| 10 | Construção dos outputs finais |
| 11 | Atualização/exportação dos resultados |
| 12 | Log final de execução |


## 1. Configuração do ambiente

Concentra todos os imports, caminhos relativos e configurações globais.

**Entrada:** nenhuma
**Saída:** variáveis de ambiente e caminhos disponíveis para todos os blocos
**Quando mexer:** ao adicionar nova biblioteca ou ajustar configurações de display.

> Todos os caminhos usam `Path.cwd()` — nenhum caminho absoluto do Mac (`/Users/...`).
> Compatível com Deepnote e outros ambientes de execução.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from __future__ import annotations

import json
import os
import warnings
from dataclasses import dataclass
from datetime import datetime
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

# ── Caminhos relativos ────────────────────────────────────────────────────────
# Todos os caminhos são relativos — sem /Users/... ou caminhos absolutos locais.
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()                               # analyses/relatorio_td/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent               # LA_Coding_Projects/
OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "relatorio_td"
DATA_DIR     = NOTEBOOK_DIR / "data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# SQL_DIR: aponta para o arquivo fonte externo (compatibilidade/debug).
# No pipeline produtivo, o SQL está embutido no Bloco 3 — não é necessário.
SQL_DIR = NOTEBOOK_DIR

# ── Configurações de display ──────────────────────────────────────────────────
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("ignore", category=FutureWarning)

DATE_TAG = datetime.now().strftime("%Y%m%d")

print(f"✅ Ambiente configurado")
print(f"   NOTEBOOK_DIR : {NOTEBOOK_DIR}")
print(f"   OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"   DATE_TAG     : {DATE_TAG}")


## 2. Parâmetros do pipeline

**Todos** os parâmetros editáveis do pipeline produtivo ficam aqui.

**Entrada:** variáveis de ambiente opcionais
**Saída:** variáveis de configuração usadas pelo pipeline inteiro
**Quando mexer:** ao alterar destino, flags de execução ou ID da planilha.

| Parâmetro | Descrição | Default |
|---|---|---|
| `PROJECT_ID` | Projeto BigQuery | `insider-data-lake` |
| `EXPORT_DEBUG_FILES` | Exporta Excel executivo de debug para `OUTPUT_DIR` | `False` |
| `UPDATE_GOOGLE_SHEETS` | Atualiza planilha existente | `True` |
| `SPREADSHEET_ID` | ID da planilha a atualizar | planilha oficial |


In [ ]:
# ── Projeto BigQuery ──────────────────────────────────────────────────────────
PROJECT_ID = os.getenv("BQ_PROJECT_ID", "insider-data-lake")

# ── Data de execução ──────────────────────────────────────────────────────────
RUN_DATE = pd.Timestamp.now(tz="America/Sao_Paulo")

# ── Flags de execução ─────────────────────────────────────────────────────────
# Se True, exporta arquivo Excel executivo de debug para OUTPUT_DIR.
EXPORT_DEBUG_FILES = False

# Se True, atualiza a planilha Google Sheets existente.
UPDATE_GOOGLE_SHEETS = True

# ── Google Sheets ─────────────────────────────────────────────────────────────
# ID da planilha EXISTENTE — nunca criar uma nova planilha.
#
# PLANILHA OFICIAL (ativa):
#    https://docs.google.com/spreadsheets/d/1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E
#
# CÓPIA DE TESTE (usar para validação):
#    https://docs.google.com/spreadsheets/d/1kUxUw-MbuAyjktkxqOQ5b9KCnprJioRJv0NTDE45aso
#
# Para alterar sem editar o notebook, defina a variável de ambiente:
#   TD_PRIORIZACAO_SPREADSHEET_ID=<ID>
SPREADSHEET_ID = os.getenv(
    "TD_PRIORIZACAO_SPREADSHEET_ID",
    "1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E",
)

# ── Thresholds da árvore de decisão lovable_priorizacao (v0) ──────────────────
# Valores hardcoded no v0. Para alterar, basta mudar aqui.
LOVABLE_THRESHOLD_TAXA_DEVOLUCAO = 0.07   # 7% de taxa de devolução
LOVABLE_THRESHOLD_TAG_CONCENTRACAO = 0.30  # 30% da tag principal sobre total de reversas

print(f"PROJECT_ID            : {PROJECT_ID}")
print(f"RUN_DATE              : {RUN_DATE}")
print(f"EXPORT_DEBUG_FILES    : {EXPORT_DEBUG_FILES}")
print(f"UPDATE_GOOGLE_SHEETS  : {UPDATE_GOOGLE_SHEETS}")
print(f"SPREADSHEET_ID        : {SPREADSHEET_ID[:24]}...")
print(f"LOVABLE_THRESHOLD_TAXA_DEV : {LOVABLE_THRESHOLD_TAXA_DEVOLUCAO:.0%}")
print(f"LOVABLE_THRESHOLD_TAG_CONC : {LOVABLE_THRESHOLD_TAG_CONCENTRACAO:.0%}")

## 3. SQL executivo e scorecard

SQL incorporado diretamente para eliminar dependência de arquivo externo.

**Quando mexer:** ao alterar filtros, janela ou regras de scoring — revisar metodologia antes.

> Não alterar thresholds, pesos ou CASE statements sem alinhamento com a equipe.
>
> O pipeline produtivo usa apenas a camada executiva por produto. A antiga base
> analítica `order × sku × tag` foi removida por ser debug-only.

### Fontes de Dados

| Tabela | Projeto | Uso |
|---|---|---|
| `business.insider_orders` | `insider-data-lake` | Pedidos válidos, período e filtros comerciais |
| `business.insider_order_items` | `insider-data-lake` | Itens de pedido e fallback de atributos |
| `fpa.analytical_dre` | `insider-data-lake` | Receita e volume agregados por produto |
| `integrated.skus` | `insider-data-lake` | Dimensão SKU e estado do SKU |
| `prepared_br.prepared__troquecommerce_order_details_br` | `insider-lake-sensitive` | Reversas ativas deduplicadas por `(order_name, id_reversa, sku)` |
| `sop_silver.return_reason_tags` | `insider-data-lake` | Tags qualitativas usadas no Top 3 PF |
| `sop_silver.portfolio_skp_clustering` | `insider-data-lake` | Cluster estratégico do produto |
| `sop_bronze.eval_produto_portfolio` | `insider-data-lake` | Pilares do scorecard de portfólio |

| Variável | Grain | Uso |
|---|---|---|
| `executive_df` | `product_name × category × gender` | Pipeline principal |
| `df_scores_raw` | `product_name` | Pilares do scorecard de portfólio |


### Query 1 — Camada executiva por produto

Executar esta query no Deepnote usando a integração `bigquery-integration`.

**Output esperado:** `executive_df`


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# INSTRUÇÃO DEEPNOTE:
# ► Converter esta célula para bigquery-integration
# ► Selecionar a conexão BigQuery do projeto
# ► Definir a variável de saída como: executive_df
# ► Executar a célula para gerar o DataFrame
# ► Grain: product_name × category × gender.
# ══════════════════════════════════════════════════════════════════════

SQL_EXECUTIVE_DF = """
-- ==============================================================================
-- PIPELINE DE ANÁLISE DE REVERSAS E PRIORIZAÇÃO DE PRODUTOS
-- ==============================================================================
--
-- Tabela executiva agregada por produto (output principal)
--
-- Fontes de dados (v2 — 2026-06-18):
--   Vendas:   business.insider_orders + business.insider_order_items
--   Reversas: prepared_br.prepared__troquecommerce_order_details_br (dedup por id_reversa)
--   Receita:  fpa.analytical_dre (agregado por product_name)
--   Tags:     sop_silver.return_reason_tags (grão order_name × sku — sem id_reversa)
--   SKU dim:  integrated.skus (fallback color/size e product_name canônico)
--   Cluster:  sop_silver.portfolio_skp_clustering (grão product_name)
--
-- Histórico de correções:
--   B1–B7 — ver versões anteriores.
--   B8 — Migração de integrated.orders/order_items → business.insider_orders/order_items.
--         Reversas dedup mudou de (order_name, sku) para (order_name, id_reversa, sku).
--         Receita e volume passaram a vir de fpa.analytical_dre (grão product_name).
-- ==============================================================================


-- ==============================================================================
-- CAMADA EXECUTIVA: TABELA AGREGADA POR PRODUTO
-- Objetivo: gerar farol decisório de priorização de melhoria física.
-- Granularidade: product_name × category × gender
-- ==============================================================================

WITH params AS (
  SELECT
    DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 12 MONTH) AS start_date,
    CURRENT_DATE("America/Sao_Paulo") AS end_date,
    30 AS min_items_vendidos,
    5 AS min_items_returned
),

-- === BASE DE VENDAS ========================================================

orders AS (
  SELECT DISTINCT
    o.order_id,
    o.order_name,
    o.processed_at,
    DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") AS data_compra,
    DATE_TRUNC(DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo"), MONTH) AS mes_compra,
    DATE_TRUNC(DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo"), WEEK(MONDAY)) AS semana_compra
  FROM `insider-data-lake.business.insider_orders` o
  CROSS JOIN params p
  WHERE o.order_status = 'paid'
    AND o.is_cancelled = FALSE
    AND (
      o.coupon_code IS NULL OR (
        NOT STARTS_WITH(o.coupon_code, 'TF-')
        AND NOT STARTS_WITH(o.coupon_code, 'TFIN')
        AND NOT STARTS_WITH(o.coupon_code, 'IR')
        AND NOT (o.coupon_code LIKE '%Item errado%')
      )
    )
    AND o.order_name IS NOT NULL
    AND o.processed_at IS NOT NULL
    AND o.store IN ('shopify_insider-world', 'shopify_insider-store-loja')
    AND DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
),

order_items AS (
  SELECT
    i.order_id,
    i.sku,
    i.product_title,
    i.variant_color,
    i.variant_size,
    i.quantity
  FROM `insider-data-lake.business.insider_order_items` i
  WHERE i.sku IS NOT NULL
),

-- Consolida itens ao nível (order_id, sku) para evitar duplicação de reversa
-- quando o mesmo SKU aparece em múltiplos registros de um mesmo pedido.
order_items_grouped AS (
  SELECT
    order_id,
    sku,
    ANY_VALUE(product_title) AS product_title,
    ANY_VALUE(variant_color) AS variant_color,
    ANY_VALUE(variant_size) AS variant_size,
    SUM(quantity) AS qt_items
  FROM order_items
  GROUP BY order_id, sku
),

sku_dim AS (
  SELECT
    sku,
    ANY_VALUE(product_name) AS product_name,
    ANY_VALUE(category) AS category,
    ANY_VALUE(gender) AS gender,
    ANY_VALUE(color) AS color,
    ANY_VALUE(size) AS size,
    ANY_VALUE(sku_state) AS sku_state
  FROM `insider-data-lake.integrated.skus`
  WHERE sku IS NOT NULL
  GROUP BY sku
),

portfolio_clustering AS (
  -- Tabela em granularidade product_name (não tem coluna sku).
  SELECT
    product_name,
    ANY_VALUE(cluster) AS portfolio_cluster,
    ANY_VALUE(TO_JSON_STRING(pc)) AS portfolio_cluster_payload
  FROM `insider-data-lake.sop_silver.portfolio_skp_clustering` pc
  GROUP BY product_name
),

-- Grain resultante: (order_id, sku) — sem GROUP BY, pois order_items_grouped
-- já consolidou os itens e todos os joins são 1:1 por SKU.
sales_item_base AS (
  SELECT
    o.order_id,
    o.order_name,
    o.data_compra,
    o.mes_compra,
    o.semana_compra,

    oi.sku,
    COALESCE(s.product_name, oi.product_title) AS product_name,
    s.category,
    s.gender,
    COALESCE(s.color, oi.variant_color) AS color,
    COALESCE(s.size, oi.variant_size) AS size,
    s.sku_state,

    oi.product_title,
    oi.variant_color,
    oi.variant_size,

    oi.qt_items
  FROM orders o
  JOIN order_items_grouped oi
    ON o.order_id = oi.order_id
  LEFT JOIN sku_dim s
    ON s.sku = oi.sku
  -- Apenas produtos perenes e lançamentos (exclui desativados e cápsulas)
  WHERE s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
),

-- === REVERSAS =============================================================

reversas_unicas AS (
  SELECT
    r.order_name,
    r.id_reversa,
    r.status,
    r.reverse_type,
    r.sku,
    r.return_reason,
    r.updated_at,
    r.created_at,
    DATE(r.created_at, "America/Sao_Paulo") AS data_reversa,
    DATE_TRUNC(DATE(r.created_at, "America/Sao_Paulo"), MONTH) AS mes_reversa,
    DATE_TRUNC(DATE(r.created_at, "America/Sao_Paulo"), WEEK(MONDAY)) AS semana_reversa,
    r.client_comment,
    SAFE_CAST(r.reverse_shipping_cost AS FLOAT64) AS reverse_shipping_cost,
    SAFE_CAST(r.retained_bonus AS FLOAT64) AS retained_bonus,
    SAFE_CAST(r.exchange_value AS FLOAT64) AS exchange_value,
    SAFE_CAST(r.refund_value AS FLOAT64) AS refund_value,

    -- COALESCE aplicado na fonte para não inflar contagens; NULL de return_quantity
    -- indica quantidade não informada no sistema (tratada como 0).
    COALESCE(SAFE_CAST(r.return_quantity AS FLOAT64), 0) AS qt_items_returned,

    CASE
      WHEN LOWER(r.return_reason) LIKE '%ficou grande%' THEN 'Tamanho'
      WHEN r.return_reason = 'Peça íntima' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Produto com defeito' THEN 'Defeito'
      WHEN r.return_reason = 'Arrependimento' THEN 'Desistência'
      WHEN r.return_reason = 'Recebi um produto errado' THEN 'Produto errado'
      WHEN LOWER(r.return_reason) LIKE '%ficou pequeno%' THEN 'Tamanho'
      WHEN r.return_reason = 'Cor diferente do esperado' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Recebi novo pedido de reposição' THEN 'Produto errado'
      WHEN r.return_reason = 'Recebi pedido duplicado' THEN 'Produto errado'
      WHEN r.return_reason = 'Arrependiemento' THEN 'Desistência'
      WHEN r.return_reason = 'Insatisfação (tamanho e cor inclusos)' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Arrependimento (tamanho e cor inclusos)' THEN 'Insatisfação com o produto'
      WHEN r.return_reason LIKE '%Tamanho%' THEN 'Tamanho'
      WHEN r.return_reason = 'Insatisfação com o produto' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Falha na Entrega' THEN 'Problema na entrega'
      WHEN r.return_reason = 'Demora na entrega' THEN 'Problema na entrega'
      WHEN r.return_reason = 'Ficou Grande' THEN 'Tamanho'
      WHEN r.return_reason = 'Ficou Pequeno' THEN 'Tamanho'
      WHEN r.return_reason = 'Não gostei do produto' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Defeito' THEN 'Defeito'
      WHEN r.return_reason = 'Peça íntima (calcinha, sutiã, meia ou cueca)' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Insatisfação' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Defeitos' THEN 'Defeito'
      WHEN r.return_reason = 'Desbotamento' THEN 'Desbotamento'
      WHEN r.return_reason = 'Desistência' THEN 'Desistência'
      WHEN r.return_reason = 'Não gostei da qualidade' THEN 'Insatisfação com o produto'
      WHEN r.return_reason IS NULL THEN NULL
      ELSE 'Outros'
    END AS motivo_classificado,

    -- Classifica o problema: Físico = atributo do produto;
    -- Logístico = falha operacional/entrega; Desistência = mudança de intenção.
    CASE
      WHEN LOWER(r.return_reason) LIKE '%ficou grande%' THEN 'Físico'
      WHEN r.return_reason = 'Peça íntima' THEN 'Físico'
      WHEN r.return_reason = 'Produto com defeito' THEN 'Físico'
      WHEN r.return_reason = 'Arrependimento' THEN 'Desistência'
      WHEN r.return_reason = 'Recebi um produto errado' THEN 'Logístico'
      WHEN LOWER(r.return_reason) LIKE '%ficou pequeno%' THEN 'Físico'
      WHEN r.return_reason = 'Cor diferente do esperado' THEN 'Físico'
      WHEN r.return_reason = 'Recebi novo pedido de reposição' THEN 'Logístico'
      WHEN r.return_reason = 'Recebi pedido duplicado' THEN 'Logístico'
      WHEN r.return_reason = 'Arrependiemento' THEN 'Desistência'
      WHEN r.return_reason = 'Insatisfação (tamanho e cor inclusos)' THEN 'Físico'
      WHEN r.return_reason = 'Arrependimento (tamanho e cor inclusos)' THEN 'Desistência'
      WHEN r.return_reason LIKE '%Tamanho%' THEN 'Físico'
      WHEN r.return_reason = 'Insatisfação com o produto' THEN 'Físico'
      WHEN r.return_reason = 'Falha na Entrega' THEN 'Logístico'
      WHEN r.return_reason = 'Demora na entrega' THEN 'Logístico'
      WHEN r.return_reason = 'Ficou Grande' THEN 'Físico'
      WHEN r.return_reason = 'Ficou Pequeno' THEN 'Físico'
      WHEN r.return_reason = 'Não gostei do produto' THEN 'Físico'
      WHEN r.return_reason = 'Defeito' THEN 'Físico'
      WHEN r.return_reason = 'Peça íntima (calcinha, sutiã, meia ou cueca)' THEN 'Físico'
      WHEN r.return_reason = 'Insatisfação' THEN 'Físico'
      WHEN r.return_reason = 'Defeitos' THEN 'Físico'
      WHEN r.return_reason = 'Desbotamento' THEN 'Físico'
      WHEN r.return_reason = 'Desistência' THEN 'Desistência'
      WHEN r.return_reason = 'Não gostei da qualidade' THEN 'Físico'
      WHEN r.return_reason IS NULL THEN NULL
      ELSE 'Outros'
    END AS tipo_problema,

    CASE
      WHEN r.reverse_type IS NULL THEN 'Sem reversa'
      WHEN LOWER(r.reverse_type) LIKE '%troca%' THEN 'Troca'
      WHEN LOWER(r.reverse_type) LIKE '%devol%' THEN 'Devolução'
      ELSE r.reverse_type
    END AS reverse_type_classificado

  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br` r
  CROSS JOIN params p
  WHERE r.status <> 'Cancelado'
    AND r.sku IS NOT NULL
    AND r.return_reason IS NOT NULL
    AND r.id_reversa IS NOT NULL
    AND DATE(r.created_at, "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY r.order_name, r.id_reversa, r.sku
    ORDER BY r.updated_at DESC
  ) = 1
),

reversas_tags_latest AS (
  SELECT
    order_name,
    sku,
    tags,
    created_at,
    DATE_TRUNC(DATE(created_at), MONTH) AS ingestion_date
  FROM `insider-data-lake.sop_silver.return_reason_tags`
  WHERE order_name IS NOT NULL
    AND sku IS NOT NULL
    AND tags IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_name, sku
    ORDER BY created_at DESC
  ) = 1
),

reversas_tag AS (
  -- Filtra tags positivas já no UNNEST para não entrarem em tag_counts.
  -- Lista em sincronia com POSITIVE_TAGS (Python) e tag_ranked NOT IN filter.
  SELECT
    order_name,
    sku,
    tag,
    ingestion_date
  FROM reversas_tags_latest,
  UNNEST(tags) AS tag
  WHERE tag NOT IN (
    -- Tags positivas
    'caimento_bom',
    'conforto_positivo',
    'feedback_positivo_geral',
    'modelagem_boa',
    'tamanho_ideal',
    'tecido_qualidade_boa',
    -- Tags logísticas/fora do escopo PF
    'atendimento_ineficiente',
    'logistica_adiantamento',
    'logistica_atraso',
    'logistica_embalagem',
    'logistica_item_errado',
    'logistica_item_faltando',
    'provador_virtual_impreciso'
  )
),

-- === MÉTRICAS POR PRODUTO =================================================

-- Receita e volume de vendas a partir do DRE (FP&A)
-- Grain: product_name (agregado de order × sku × atribuição)
-- JOIN com orders para herdar filtro de período; JOIN com sku_dim para product_name
dre_product_metrics AS (
  SELECT
    s.product_name,
    SUM(dre.non_refunded_quantity) AS qt_items_vendidos,
    SUM(dre.revenue_after_discounts) AS receita_liquida
  FROM `insider-data-lake.fpa.analytical_dre` dre
  JOIN orders o
    ON dre.order_id = o.order_id
  JOIN sku_dim s
    ON dre.sku = s.sku
  WHERE s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
  GROUP BY s.product_name
),

sales_product_metrics AS (
  SELECT
    sib.product_name,
    sib.category,
    sib.gender,
    ANY_VALUE(pc.portfolio_cluster) AS portfolio_cluster,
    ANY_VALUE(pc.portfolio_cluster_payload) AS portfolio_cluster_payload,

    COUNT(DISTINCT sib.order_id) AS qt_pedidos,
    COUNT(DISTINCT sib.sku) AS qt_skus,
    -- Volume e receita: DRE como fonte de verdade; fallback para order_items se DRE vazio
    COALESCE(ANY_VALUE(dpm.qt_items_vendidos), SUM(sib.qt_items)) AS qt_items_vendidos,
    COALESCE(ANY_VALUE(dpm.receita_liquida), 0) AS receita_liquida
  FROM sales_item_base sib
  LEFT JOIN portfolio_clustering pc
    ON pc.product_name = sib.product_name
  LEFT JOIN dre_product_metrics dpm
    ON dpm.product_name = sib.product_name
  GROUP BY sib.product_name, sib.category, sib.gender
),

-- Join de reversas às vendas na grain (order_name, sku).
-- Tags NÃO entram aqui para não inflar contagens de T&D.
-- Nota: com dedup por id_reversa, pode haver múltiplas linhas por (order_name, sku)
-- quando existem múltiplas reversas para o mesmo item. COUNT DISTINCT protege as contagens.
td_joined AS (
  SELECT
    ru.order_name,
    ru.sku,
    sib.product_name,
    sib.category,
    sib.gender,
    sib.color,
    sib.size,

    ru.data_reversa,
    ru.mes_reversa,
    ru.reverse_type_classificado,
    ru.return_reason,
    ru.motivo_classificado,
    ru.tipo_problema,
    ru.client_comment,

    ru.qt_items_returned,
    ru.exchange_value,
    ru.refund_value
  FROM reversas_unicas ru
  JOIN sales_item_base sib
    ON ru.order_name = sib.order_name
   AND ru.sku = sib.sku
),

td_product_metrics AS (
  SELECT
    product_name,
    category,
    gender,

    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas,
    SUM(qt_items_returned) AS qt_items_returned,

    COUNT(DISTINCT IF(reverse_type_classificado = 'Troca', CONCAT(order_name, '|', sku), NULL)) AS qt_trocas,
    COUNT(DISTINCT IF(reverse_type_classificado = 'Devolução', CONCAT(order_name, '|', sku), NULL)) AS qt_devolucoes,

    COUNT(DISTINCT IF(tipo_problema = 'Físico', CONCAT(order_name, '|', sku), NULL)) AS qt_reversas_fisico,
    COUNT(DISTINCT IF(tipo_problema = 'Logístico', CONCAT(order_name, '|', sku), NULL)) AS qt_reversas_logistico,

    SUM(exchange_value) AS valor_troca,
    SUM(refund_value) AS valor_devolucao
  FROM td_joined
  GROUP BY product_name, category, gender
),

category_metrics AS (
  SELECT
    spm.category,
    SUM(spm.qt_items_vendidos) AS qt_items_vendidos_categoria,
    SUM(COALESCE(tpm.qt_items_returned, 0)) AS qt_items_returned_categoria,
    SAFE_DIVIDE(
      SUM(COALESCE(tpm.qt_items_returned, 0)),
      SUM(spm.qt_items_vendidos)
    ) AS td_rate_categoria
  FROM sales_product_metrics spm
  LEFT JOIN td_product_metrics tpm
    ON spm.product_name = tpm.product_name
   AND spm.category = tpm.category
   AND spm.gender = tpm.gender
  GROUP BY spm.category
),

-- qt_reversas_tag: COUNT DISTINCT (order_name|sku) por tag — não infla T&D.
-- Uma reversa com N tags contribui com 1 para o contador de cada tag.
tag_counts AS (
  SELECT
    tj.product_name,
    tj.category,
    tj.gender,
    rt.tag AS problema_tag,
    COUNT(DISTINCT CONCAT(tj.order_name, '|', tj.sku)) AS qt_reversas_tag
  FROM td_joined tj
  JOIN reversas_tag rt
    ON rt.order_name = tj.order_name
   AND rt.sku = tj.sku
  WHERE rt.tag IS NOT NULL
  GROUP BY tj.product_name, tj.category, tj.gender, rt.tag
),

-- Denominador correto para pct de tags: total de reversas distintas do produto,
-- independente de terem ou não tags. Evita que pct_top_3_total ultrapasse 100%.
td_reversas_total AS (
  SELECT
    product_name,
    category,
    gender,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas_total
  FROM td_joined
  GROUP BY product_name, category, gender
),

tag_ranked AS (
  SELECT
    tc.*,
    rt.qt_reversas_total,
    ROW_NUMBER() OVER (
      PARTITION BY tc.product_name, tc.category, tc.gender
      ORDER BY tc.qt_reversas_tag DESC, tc.problema_tag
    ) AS tag_rank
  FROM tag_counts tc
  JOIN td_reversas_total rt
    ON tc.product_name = rt.product_name
   AND tc.category = rt.category
   AND tc.gender = rt.gender
),

top_tags AS (
  SELECT
    product_name,
    category,
    gender,

    MAX(IF(tag_rank = 1, problema_tag, NULL)) AS top_1_problema,
    MAX(IF(tag_rank = 1, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_1_pct,

    MAX(IF(tag_rank = 2, problema_tag, NULL)) AS top_2_problema,
    MAX(IF(tag_rank = 2, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_2_pct,

    MAX(IF(tag_rank = 3, problema_tag, NULL)) AS top_3_problema,
    MAX(IF(tag_rank = 3, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_3_pct,

    MAX(IF(tag_rank = 4, problema_tag, NULL)) AS top_4_problema,
    MAX(IF(tag_rank = 4, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_4_pct,

    MAX(IF(tag_rank = 5, problema_tag, NULL)) AS top_5_problema,
    MAX(IF(tag_rank = 5, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_5_pct,

    SAFE_DIVIDE(
      SUM(IF(tag_rank <= 3, qt_reversas_tag, 0)),
      MAX(qt_reversas_total)
    ) AS pct_top_3_total
  FROM tag_ranked
  WHERE tag_rank <= 5
  GROUP BY product_name, category, gender
),

color_concentration AS (
  SELECT
    product_name,
    category,
    gender,
    color,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas_color,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) / SUM(COUNT(DISTINCT CONCAT(order_name, '|', sku))) OVER (
      PARTITION BY product_name, category, gender
    ) AS pct_reversas_color
  FROM td_joined
  WHERE color IS NOT NULL
  GROUP BY product_name, category, gender, color
),

main_color AS (
  SELECT
    product_name,
    category,
    gender,
    color AS principal_cor_afetada,
    pct_reversas_color AS principal_cor_pct
  FROM color_concentration
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_name, category, gender
    ORDER BY pct_reversas_color DESC, color
  ) = 1
),

size_concentration AS (
  SELECT
    product_name,
    category,
    gender,
    size,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas_size,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) / SUM(COUNT(DISTINCT CONCAT(order_name, '|', sku))) OVER (
      PARTITION BY product_name, category, gender
    ) AS pct_reversas_size
  FROM td_joined
  WHERE size IS NOT NULL
  GROUP BY product_name, category, gender, size
),

main_size AS (
  SELECT
    product_name,
    category,
    gender,
    size AS principal_tamanho_afetado,
    pct_reversas_size AS principal_tamanho_pct
  FROM size_concentration
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_name, category, gender
    ORDER BY pct_reversas_size DESC, size
  ) = 1
),

comments_sample AS (
  SELECT
    product_name,
    category,
    gender,
    STRING_AGG(client_comment, ' || ' LIMIT 10) AS comentarios_amostra
  FROM (
    SELECT DISTINCT
      product_name,
      category,
      gender,
      client_comment
    FROM td_joined
    WHERE client_comment IS NOT NULL
      AND LENGTH(TRIM(client_comment)) > 0
  )
  GROUP BY product_name, category, gender
),

trend_base AS (
  SELECT
    product_name,
    category,
    gender,

    COUNT(DISTINCT IF(
      data_reversa >= DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 3 MONTH),
      CONCAT(order_name, '|', sku),
      NULL
    )) AS reversas_ultimos_3m,

    COUNT(DISTINCT IF(
      data_reversa >= DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 6 MONTH)
      AND data_reversa < DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 3 MONTH),
      CONCAT(order_name, '|', sku),
      NULL
    )) AS reversas_3m_anteriores
  FROM td_joined
  GROUP BY product_name, category, gender
),

trend_classified AS (
  SELECT
    *,
    CASE
      WHEN reversas_ultimos_3m + reversas_3m_anteriores < 5 THEN 'Sem volume para tendência'
      WHEN reversas_3m_anteriores = 0 AND reversas_ultimos_3m > 0 THEN 'Apareceu nos últimos 3 meses'
      WHEN SAFE_DIVIDE(reversas_ultimos_3m - reversas_3m_anteriores, reversas_3m_anteriores) >= 0.25 THEN 'Aumentou nos últimos 3 meses'
      WHEN SAFE_DIVIDE(reversas_ultimos_3m - reversas_3m_anteriores, reversas_3m_anteriores) <= -0.25 THEN 'Caiu nos últimos 3 meses'
      ELSE 'Estável'
    END AS tendencia_reversas
  FROM trend_base
),

-- ─── Sell-through: mapeamento SKU → product_name ─────────────────────────────────────

sku_map_st AS (
  SELECT DISTINCT
    product_name,
    sku
  FROM `insider-data-lake.integrated.skus`
  WHERE product_name IS NOT NULL
    AND sku IS NOT NULL
),

-- ─── Sell-through: primeira data de venda por produto ────────────────────────────────

first_sale AS (
  SELECT
    product_name,
    ANY_VALUE(first_sale_date) AS first_sale_date
  FROM `insider-data-lake.sop_silver.portfolio_skp_clustering`
  WHERE first_sale_date IS NOT NULL
  GROUP BY product_name
),

-- ─── Sell-through: unidades vendidas em janelas de 30 / 60 / 90 dias ─────────────────

sales_windows AS (
  SELECT
    sm.product_name,
    SUM(CASE
      WHEN DATE(o.processed_at, 'America/Sao_Paulo')
             BETWEEN fs.first_sale_date
             AND DATE_ADD(fs.first_sale_date, INTERVAL 30 DAY)
      THEN oi.quantity ELSE 0
    END) AS sold_30d,
    SUM(CASE
      WHEN DATE(o.processed_at, 'America/Sao_Paulo')
             BETWEEN fs.first_sale_date
             AND DATE_ADD(fs.first_sale_date, INTERVAL 60 DAY)
      THEN oi.quantity ELSE 0
    END) AS sold_60d,
    SUM(CASE
      WHEN DATE(o.processed_at, 'America/Sao_Paulo')
             BETWEEN fs.first_sale_date
             AND DATE_ADD(fs.first_sale_date, INTERVAL 90 DAY)
      THEN oi.quantity ELSE 0
    END) AS sold_90d
  FROM sku_map_st sm
  JOIN first_sale fs ON sm.product_name = fs.product_name
  JOIN `insider-data-lake.business.insider_order_items` oi ON sm.sku = oi.sku
  JOIN `insider-data-lake.business.insider_orders` o ON oi.order_id = o.order_id
  WHERE o.is_cancelled = FALSE
    AND o.order_status = 'paid'
  GROUP BY 1
),

-- ─── Sell-through: estoque físico no dia de primeira venda ───────────────────────────

launch_stock AS (
  SELECT
    sm.product_name,
    SUM(st.physical_stock) AS initial_stock
  FROM sku_map_st sm
  JOIN first_sale fs ON sm.product_name = fs.product_name
  JOIN `insider-data-lake.integrated.stock` st
    ON sm.sku = st.sku
   AND st.stock_date = fs.first_sale_date
  GROUP BY 1
),

-- ─── Sell-through: razão vendas / estoque inicial ────────────────────────────────────

sell_through AS (
  SELECT
    sw.product_name,
    SAFE_DIVIDE(sw.sold_30d, ls.initial_stock) AS sell_through_30d,
    SAFE_DIVIDE(sw.sold_60d, ls.initial_stock) AS sell_through_60d,
    SAFE_DIVIDE(sw.sold_90d, ls.initial_stock) AS sell_through_90d
  FROM sales_windows sw
  LEFT JOIN launch_stock ls ON sw.product_name = ls.product_name
),

-- ─── MC3: campos diretos de eval_produto_portfolio ───────────────────────────────────

mc3_base AS (
  SELECT
    product_name,
    ANY_VALUE(mc3_ratio) AS mc3_ratio,
    ANY_VALUE(mc3_ratio_cat4) AS mc3_ratio_cat4,
    ANY_VALUE(mc3_ratio_portfolio) AS mc3_ratio_portfolio,
    ANY_VALUE(net_profit_after_marketing_costs) AS net_profit_after_marketing_costs
  FROM `insider-data-lake.sop_bronze.eval_produto_portfolio`
  WHERE product_name IS NOT NULL
  GROUP BY product_name
),

-- === SCORING ==============================================================

percentiles AS (
  SELECT
    *,

    CUME_DIST() OVER (ORDER BY receita_liquida) AS percentil_receita,
    CUME_DIST() OVER (ORDER BY qt_items_vendidos) AS percentil_unidades,
    CUME_DIST() OVER (ORDER BY td_rate) AS percentil_td_rate,
    CUME_DIST() OVER (ORDER BY qt_items_returned) AS percentil_volume_td,
    CUME_DIST() OVER (ORDER BY delta_vs_categoria) AS percentil_delta_vs_categoria,

    -- Percentis de sell-through (NULL se sem cobertura — fallback no scored CTE)
    CUME_DIST() OVER (ORDER BY sell_through_30d) AS percentil_sell_through_30d,
    CUME_DIST() OVER (ORDER BY sell_through_60d) AS percentil_sell_through_60d,
    CUME_DIST() OVER (ORDER BY sell_through_90d) AS percentil_sell_through_90d,

    -- Receita média mensal vs categoria: percentil particionado por category
    CUME_DIST() OVER (
      PARTITION BY category
      ORDER BY receita_media_mensal_vs_categoria
    ) AS percentil_receita_vs_categoria,

    -- Sub-scores MC3 (escala discreta 0.25 / 0.50 / 0.75 / 1.00)
    CASE
      WHEN mc3_ratio IS NULL OR mc3_ratio_cat4 IS NULL THEN NULL
      WHEN mc3_ratio >= mc3_ratio_cat4             THEN 1.00
      WHEN mc3_ratio >= 0.90 * mc3_ratio_cat4      THEN 0.75
      WHEN mc3_ratio >= 0.75 * mc3_ratio_cat4      THEN 0.50
      ELSE 0.25
    END AS mc3_vs_categoria_score,

    CASE
      WHEN mc3_ratio IS NULL OR mc3_ratio_portfolio IS NULL THEN NULL
      WHEN mc3_ratio >= mc3_ratio_portfolio          THEN 1.00
      WHEN mc3_ratio >= 0.90 * mc3_ratio_portfolio   THEN 0.75
      WHEN mc3_ratio >= 0.75 * mc3_ratio_portfolio   THEN 0.50
      ELSE 0.25
    END AS mc3_vs_portfolio_score,

    CUME_DIST() OVER (ORDER BY net_profit_after_marketing_costs) AS representatividade_mc3_score

  FROM (
    SELECT
      spm.product_name,
      spm.category,
      spm.gender,
      spm.portfolio_cluster,
      spm.portfolio_cluster_payload,

      spm.qt_pedidos,
      spm.qt_skus,
      spm.qt_items_vendidos,
      spm.receita_liquida,

      COALESCE(tpm.qt_reversas, 0) AS qt_reversas,
      COALESCE(tpm.qt_items_returned, 0) AS qt_items_returned,
      COALESCE(tpm.qt_trocas, 0) AS qt_trocas,
      COALESCE(tpm.qt_devolucoes, 0) AS qt_devolucoes,
      COALESCE(tpm.qt_reversas_fisico, 0) AS qt_reversas_fisico,
      COALESCE(tpm.qt_reversas_logistico, 0) AS qt_reversas_logistico,
      COALESCE(tpm.valor_troca, 0) AS valor_troca,
      COALESCE(tpm.valor_devolucao, 0) AS valor_devolucao,

      SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), spm.qt_items_vendidos) AS td_rate,

      cm.td_rate_categoria,
      SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), spm.qt_items_vendidos) - cm.td_rate_categoria AS delta_vs_categoria,
      SAFE_DIVIDE(
        SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), spm.qt_items_vendidos),
        cm.td_rate_categoria
      ) AS ratio_vs_categoria,

      SAFE_DIVIDE(spm.receita_liquida, SUM(spm.receita_liquida) OVER ()) AS share_receita_portfolio,
      SAFE_DIVIDE(spm.qt_items_vendidos, SUM(spm.qt_items_vendidos) OVER ()) AS share_unidades_portfolio,
      SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), SUM(COALESCE(tpm.qt_items_returned, 0)) OVER ()) AS share_td_portfolio,

      tt.top_1_problema,
      tt.top_1_pct,
      tt.top_2_problema,
      tt.top_2_pct,
      tt.top_3_problema,
      tt.top_3_pct,
      tt.top_4_problema,
      tt.top_4_pct,
      tt.top_5_problema,
      tt.top_5_pct,
      tt.pct_top_3_total,

      mc.principal_cor_afetada,
      mc.principal_cor_pct,
      ms.principal_tamanho_afetado,
      ms.principal_tamanho_pct,

      tb.reversas_ultimos_3m,
      tb.reversas_3m_anteriores,
      tb.tendencia_reversas,

      cs.comentarios_amostra,

      -- Sell-through (NULL quando sem cobertura de SKU/estoque)
      st.sell_through_30d,
      st.sell_through_60d,
      st.sell_through_90d,

      -- Receita média mensal vs média da categoria (ratio; usado para CUME_DIST)
      SAFE_DIVIDE(
        spm.receita_liquida,
        NULLIF(AVG(spm.receita_liquida) OVER (PARTITION BY spm.category), 0)
      ) AS receita_media_mensal_vs_categoria,

      -- MC3 fields (NULL quando produto não está em eval_produto_portfolio)
      mb.mc3_ratio,
      mb.mc3_ratio_cat4,
      mb.mc3_ratio_portfolio,
      mb.net_profit_after_marketing_costs

    FROM sales_product_metrics spm
    LEFT JOIN td_product_metrics tpm
      ON spm.product_name = tpm.product_name
     AND spm.category = tpm.category
     AND spm.gender = tpm.gender
    LEFT JOIN category_metrics cm
      ON spm.category = cm.category
    LEFT JOIN top_tags tt
      ON spm.product_name = tt.product_name
     AND spm.category = tt.category
     AND spm.gender = tt.gender
    LEFT JOIN main_color mc
      ON spm.product_name = mc.product_name
     AND spm.category = mc.category
     AND spm.gender = mc.gender
    LEFT JOIN main_size ms
      ON spm.product_name = ms.product_name
     AND spm.category = ms.category
     AND spm.gender = ms.gender
    LEFT JOIN trend_classified tb
      ON spm.product_name = tb.product_name
     AND spm.category = tb.category
     AND spm.gender = tb.gender
    LEFT JOIN comments_sample cs
      ON spm.product_name = cs.product_name
     AND spm.category = cs.category
     AND spm.gender = cs.gender
    LEFT JOIN sell_through st
      ON spm.product_name = st.product_name
    LEFT JOIN mc3_base mb
      ON spm.product_name = mb.product_name
  )
),
  )
scored AS (
  SELECT
    *,
  SELECT
    0.5 * percentil_td_rate
      + 0.3 * percentil_volume_td
      + 0.2 * percentil_delta_vs_categoria AS td_score,

    -- commercial_score_v2 = 0.70 × tracao_vendas_score + 0.30 × mc3_score.
    -- Fallback (a): sell-through indisponível → usa percentil_receita/percentil_unidades.
    -- Fallback (b): mc3 indisponível → usa apenas tracao_vendas_score com peso total.
    CASE
      WHEN mc3_vs_categoria_score IS NOT NULL
      THEN
        0.70 * COALESCE(
            0.30 * percentil_sell_through_30d
              + 0.30 * percentil_sell_through_60d
              + 0.25 * percentil_sell_through_90d
              + 0.15 * percentil_receita_vs_categoria,
            0.60 * percentil_receita + 0.40 * percentil_unidades
          )
          + 0.30 * (
              0.50 * mc3_vs_categoria_score
                + 0.30 * mc3_vs_portfolio_score
                + 0.20 * representatividade_mc3_score
            )
      ELSE
        COALESCE(
          0.30 * percentil_sell_through_30d
            + 0.30 * percentil_sell_through_60d
            + 0.25 * percentil_sell_through_90d
            + 0.15 * percentil_receita_vs_categoria,
          0.60 * percentil_receita + 0.40 * percentil_unidades
        )
    END AS commercial_score_v2,

    -- priority_score expande td_score e commercial_score_v2 inline (BigQuery não
    -- permite referência a alias da mesma cláusula SELECT).
    0.50 * (
      0.5 * percentil_td_rate
        + 0.3 * percentil_volume_td
        + 0.2 * percentil_delta_vs_categoria
    )
    + 0.50 * (
      CASE
        WHEN mc3_vs_categoria_score IS NOT NULL
        THEN
          0.70 * COALESCE(
              0.30 * percentil_sell_through_30d
                + 0.30 * percentil_sell_through_60d
                + 0.25 * percentil_sell_through_90d
                + 0.15 * percentil_receita_vs_categoria,
              0.60 * percentil_receita + 0.40 * percentil_unidades
            )
            + 0.30 * (
                0.50 * mc3_vs_categoria_score
                  + 0.30 * mc3_vs_portfolio_score
                  + 0.20 * representatividade_mc3_score
              )
        ELSE
          COALESCE(
            0.30 * percentil_sell_through_30d
              + 0.30 * percentil_sell_through_60d
              + 0.25 * percentil_sell_through_90d
              + 0.15 * percentil_receita_vs_categoria,
            0.60 * percentil_receita + 0.40 * percentil_unidades
          )
      END
    ) AS priority_score
  FROM percentiles
),
  FROM percentiles
final AS (
  SELECT
    *,

    CASE
      -- Volume mínimo estatístico verificado primeiro — guarda contra ruído.
      WHEN qt_items_vendidos < (SELECT min_items_vendidos FROM params)
        OR qt_items_returned < (SELECT min_items_returned FROM params)
        THEN 'Sem evidência suficiente'

      -- Alta dor + relevância comercial → ação imediata.
      WHEN td_score >= 0.70
        AND commercial_score_v2 >= 0.60
        THEN 'Priorizar melhoria'

      -- Alta dor, baixa tração comercial → vigilância.
      WHEN td_score >= 0.70
        AND commercial_score_v2 < 0.60
        THEN 'Monitorar'

      -- Produto de alta relevância com dor moderada → alerta preventivo.
      WHEN commercial_score_v2 >= 0.70
        AND td_score >= 0.50
        AND td_score < 0.70
        THEN 'Alerta em produto relevante'

      -- Dor baixa ou produto irrelevante → não priorizar.
      WHEN td_score < 0.50
        OR commercial_score_v2 < 0.40
        THEN 'Não priorizar agora'

      -- Zona cinza: td_score ∈ [0.50, 0.70) e commercial_score_v2 ∈ [0.40, 0.70).
      -- Dor e tração medianas — acompanhar sem ação imediata.
      ELSE 'Monitorar'
    END AS sinal_priorizacao,

    CONCAT(
      'Top problemas: ',
      COALESCE(top_1_problema, 'sem tag'),
      IF(top_2_problema IS NOT NULL, CONCAT(', ', top_2_problema), ''),
      IF(top_3_problema IS NOT NULL, CONCAT(', ', top_3_problema), ''),
      '. ',
      'Top 3 concentram ',
      CAST(ROUND(100 * COALESCE(pct_top_3_total, 0), 1) AS STRING),
      '% das reversas com tag. ',
      IF(
        principal_cor_afetada IS NOT NULL AND principal_cor_pct >= 0.50,
        CONCAT('Há concentração relevante na cor ', principal_cor_afetada, '. '),
        ''
      ),
      IF(
        principal_tamanho_afetado IS NOT NULL AND principal_tamanho_pct >= 0.50,
        CONCAT('Há concentração relevante no tamanho ', principal_tamanho_afetado, '. '),
        ''
      ),
      IF(
        tendencia_reversas IS NOT NULL,
        CONCAT('Tendência: ', tendencia_reversas, '.'),
        ''
      )
    ) AS resumo_pre_llm

  FROM scored
)

SELECT
  product_name,
  category,
  gender,
  portfolio_cluster,

  sinal_priorizacao,
  priority_score,
  td_score,
  commercial_score_v2,

  qt_pedidos,
  qt_skus,
  qt_items_vendidos,
  receita_liquida,

  qt_reversas,
  qt_items_returned,
  qt_trocas,
  qt_devolucoes,
  qt_reversas_fisico,
  qt_reversas_logistico,
  valor_troca,
  valor_devolucao,

  td_rate,
  td_rate_categoria,
  delta_vs_categoria,
  ratio_vs_categoria,

  share_receita_portfolio,
  share_unidades_portfolio,
  share_td_portfolio,

  top_1_problema,
  top_1_pct,
  top_2_problema,
  top_2_pct,
  top_3_problema,
  top_3_pct,
  top_4_problema,
  top_4_pct,
  top_5_problema,
  top_5_pct,
  pct_top_3_total,

  principal_cor_afetada,
  principal_cor_pct,
  principal_tamanho_afetado,
  principal_tamanho_pct,

  reversas_ultimos_3m,
  reversas_3m_anteriores,
  tendencia_reversas,

  comentarios_amostra,
  resumo_pre_llm,
  portfolio_cluster_payload

FROM final
ORDER BY
  CASE sinal_priorizacao
    WHEN 'Priorizar melhoria' THEN 1
    WHEN 'Alerta em produto relevante' THEN 2
    WHEN 'Monitorar' THEN 3
    WHEN 'Sem evidência suficiente' THEN 4
    WHEN 'Não priorizar agora' THEN 5
    ELSE 6
  END,
  priority_score DESC,
  qt_items_returned DESC;
"""
  qt_items_returned DESC;
"""

"""


  priority_score DESC,  qt_items_returned DESC;
"""


"""  priority_score DESC,  qt_items_returned DESC;


### Query 2 — Scorecard de portfólio por produto

Executar esta query no Deepnote usando a integração `bigquery-integration`.

**Output esperado:** `df_scores_raw`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# INSTRUÇÃO DEEPNOTE:
# ► Converter esta célula para bigquery-integration
# ► Selecionar a conexão BigQuery do projeto
# ► Definir a variável de saída como: df_scores_raw
# ► Executar a célula para gerar o DataFrame
# ► Grain: product_name. Fonte: sop_bronze.eval_produto_portfolio.
# ══════════════════════════════════════════════════════════════════════

SQL_DF_SCORES_RAW = """
SELECT
    product_name,
    ROUND(ANY_VALUE(score_vendas_geral) * 100, 1)           AS score_tracao_comercial,
    ROUND(ANY_VALUE(score_viabilidade_financeira) * 100, 1)  AS score_unit_economics,
    ROUND(ANY_VALUE(score_satisf_cliente) * 100, 1)          AS score_satisfacao_marca
  FROM `insider-data-lake.sop_bronze.eval_produto_portfolio`
  GROUP BY product_name
"""


## 4. Pós-extração dos DataFrames fonte

**Entrada:** `executive_df`, `df_scores_raw`
**Saída:** logs de shape/colunas e DataFrames prontos para preparação
**Quando mexer:** ao alterar os outputs das queries fonte.


In [ ]:
# ── Pós-extração: executive_df ────────────────────────────────────────────────
# Executar após a célula bigquery-integration acima ter gerado executive_df.
print(f"\n✅ executive_df: {executive_df.shape[0]:,} linhas × {executive_df.shape[1]:,} colunas")
print(f"   Colunas: {list(executive_df.columns)}")
display(executive_df.head(3))


In [ ]:
# ── Pós-extração: Scorecard (df_scores_raw) ──────────────────────────────────
# Executar após a célula bigquery-integration acima ter gerado df_scores_raw.
print(f"\n✅ df_scores_raw: {df_scores_raw.shape[0]:,} produto(s) com scorecard")
print(f"   Colunas: {list(df_scores_raw.columns)}")


## 5. Funções utilitárias e regras de negócio

Funções incorporadas de `td_analysis_functions.py` para eliminar dependência do arquivo externo.

**Entrada:** nenhuma (definições)
**Saída:** constantes, dataclasses e funções disponíveis para os blocos seguintes
**Quando mexer:** ao corrigir bugs — manter `td_analysis_functions.py` em sincronia.

> Não alterar thresholds, labels, mapeamentos de tags ou lógica de classificação sem revisão.


### 5.1 — Constantes, dataclasses e mapeamentos de negócio

`PrioritizationThresholds`, `TAG_TO_ACTION`, `POSITIVE_TAGS`, `LOGISTICA_TAGS`,
`TOP_PROBLEMAS_EXCLUDED_TAGS`, `EXECUTIVE_REQUIRED_COLUMNS`.


In [ ]:
"""Utility functions for analyzing Insider Store T&D prioritization outputs.

Expected input
--------------
Executive product-level dataframe generated by the T&D prioritization query.

The functions are intentionally pandas-only and side-effect free, so they can be
used in notebooks, scripts, Streamlit, or scheduled checks.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd


EXECUTIVE_REQUIRED_COLUMNS = {
    "product_name",
    "category",
    "sinal_priorizacao",
    "priority_score",
    "td_score",
    "commercial_score_v2",
    "qt_items_vendidos",
    "receita_liquida",
    "qt_items_returned",
    "td_rate",
    "td_rate_categoria",
    "delta_vs_categoria",
}

# Name of the commercial-score column produced by the SQL pipeline.
# v2 = 0.70 × tracao_vendas_score + 0.30 × mc3_score (with fallbacks).
COMMERCIAL_SCORE_COL: str = "commercial_score_v2"

@dataclass(frozen=True)
class PrioritizationThresholds:
    """Thresholds used to classify products in Python.

    Keep these aligned with the SQL params and CASE statement unless you are
    deliberately running a sensitivity analysis.
    """

    min_items_vendidos: int = 30
    min_items_returned: int = 5
    td_score_prioritize: float = 0.70
    commercial_score_prioritize: float = 0.60
    commercial_score_alert: float = 0.70
    td_score_alert_min: float = 0.50
    low_td_score: float = 0.50
    low_commercial_score: float = 0.40


# ---------------------------------------------------------------------------
# Tag → action type mapping
# ---------------------------------------------------------------------------

TAG_TO_ACTION: dict[str, str] = {
    # Modelagem
    "caimento_ruim": "Modelagem",
    "modelagem_ruim": "Modelagem",
    "sustentacao_ruim": "Modelagem",
    # Grade
    "comprimento_curto": "Grade",
    "comprimento_longo": "Grade",
    "tamanho_grande": "Grade",
    "tamanho_pequeno": "Grade",
    "tamanho_pepequeno": "Grade",
    # Tecido
    "tecido_fino": "Tecido",
    "tecido_grosso": "Tecido",
    "tecido_transparente": "Tecido",
    "tecido_qualidade_ruim": "Tecido",
    "tecido_marca_corpo": "Tecido",
    "tecido_quente": "Tecido",
    "tecido_amassa": "Tecido",
    "pilling_bolinhas": "Tecido",
    "encolhimento": "Tecido",
    # Defeito → Investigação adicional
    "defeito_costura": "Investigação adicional",
    "defeito_aviamento": "Investigação adicional",
    "defeito_fio_puxado": "Investigação adicional",
    "defeito_furo_rasgo": "Investigação adicional",
    "defeito_gola": "Investigação adicional",
    "defeito_mancha": "Investigação adicional",
    # Ambíguo → Investigação adicional
    "conforto_negativo": "Investigação adicional",
    # Comunicação
    "cor_diferente_site": "PDP/Comunicação",
}

POSITIVE_TAGS: frozenset[str] = frozenset({
    "caimento_bom",
    "conforto_positivo",
    "feedback_positivo_geral",
    "modelagem_boa",
    "tamanho_ideal",
    "tecido_qualidade_boa",
})

LOGISTICA_TAGS: frozenset[str] = frozenset({
    "atendimento_ineficiente",
    "logistica_adiantamento",
    "logistica_atraso",
    "logistica_embalagem",
    "logistica_item_errado",
    "logistica_item_faltando",
    "provador_virtual_impreciso",
})

TOP_PROBLEMAS_EXCLUDED_TAGS: frozenset[str] = POSITIVE_TAGS | LOGISTICA_TAGS

# ---------------------------------------------------------------------------
# Workbook formatting constants (openpyxl)
# ---------------------------------------------------------------------------

_TAB_COLORS: dict[str, str] = {
    "farol_executivo": "4472C4",
    "resumo_farol": "70AD47",
    "benchmark_categoria": "9DC3E6",
    "resumo_tags": "9DC3E6",
    "priorizar_melhoria": "C00000",
    "relatorio_pf": "ED7D31",
}

_SINAL_ROW_COLORS: dict[str, str] = {
    "Priorizar melhoria": "FFD7D7",
    "Alerta em produto relevante": "FFF9D7",
    "Monitorar": "FFE8CC",
    "Não priorizar agora": "F2F2F2",
    "Sem evidência suficiente": "FFFFFF",
}


def validate_columns(
    df: pd.DataFrame,
    required_columns: Iterable[str],
    dataframe_name: str = "dataframe",
) -> None:
    """Raise a clear error if required columns are missing."""

    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(
            f"{dataframe_name} is missing required columns: {', '.join(missing)}"
        )


def coerce_numeric(
    df: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    """Return a copy with selected columns converted to numeric values."""

    out = df.copy()
    for column in columns:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors="coerce")
    return out


def prepare_executive_df(df: pd.DataFrame) -> pd.DataFrame:
    """Validate and normalize the executive product-level dataframe."""

    validate_columns(df, EXECUTIVE_REQUIRED_COLUMNS, "executive_df")

    numeric_columns = [
        "priority_score",
        "td_score",
        "commercial_score_v2",
        "qt_pedidos",
        "qt_skus",
        "qt_items_vendidos",
        "receita_liquida",
        "qt_reversas",
        "qt_items_returned",
        "qt_trocas",
        "qt_devolucoes",
        "valor_troca",
        "valor_devolucao",
        "td_rate",
        "td_rate_categoria",
        "delta_vs_categoria",
        "ratio_vs_categoria",
        "share_receita_portfolio",
        "share_unidades_portfolio",
        "share_td_portfolio",
        "top_1_pct",
        "top_2_pct",
        "top_3_pct",
        "pct_top_3_total",
        "principal_cor_pct",
        "principal_tamanho_pct",
        "reversas_ultimos_3m",
        "reversas_3m_anteriores",
        "qt_reversas_fisico",
        "qt_reversas_logistico",
    ]
    out = coerce_numeric(df, numeric_columns)

    sort_cols = ["priority_score", "qt_items_returned", "receita_liquida"]
    available_sort_cols = [c for c in sort_cols if c in out.columns]
    return out.sort_values(available_sort_cols, ascending=False).reset_index(drop=True)


def classify_priority(
    df: pd.DataFrame,
    thresholds: PrioritizationThresholds = PrioritizationThresholds(),
) -> pd.DataFrame:
    """Recompute prioritization labels in Python for QA or sensitivity tests."""

    out = prepare_executive_df(df)

    conditions = [
        (out["qt_items_vendidos"] < thresholds.min_items_vendidos)
        | (out["qt_items_returned"] < thresholds.min_items_returned),
        (out["td_score"] >= thresholds.td_score_prioritize)
        & (out[COMMERCIAL_SCORE_COL] >= thresholds.commercial_score_prioritize),
        (out["td_score"] >= thresholds.td_score_prioritize)
        & (out[COMMERCIAL_SCORE_COL] < thresholds.commercial_score_prioritize),
        (out[COMMERCIAL_SCORE_COL] >= thresholds.commercial_score_alert)
        & (out["td_score"] >= thresholds.td_score_alert_min)
        & (out["td_score"] < thresholds.td_score_prioritize),
        (out["td_score"] < thresholds.low_td_score)
        | (out[COMMERCIAL_SCORE_COL] < thresholds.low_commercial_score),
    ]
    labels = [
        "Sem evidência suficiente",
        "Priorizar melhoria",
        "Monitorar",
        "Alerta em produto relevante",
        "Não priorizar agora",
    ]

    out["sinal_priorizacao_python"] = np.select(conditions, labels, default="Monitorar")
    out["sinal_confere_sql"] = out["sinal_priorizacao_python"].eq(
        out["sinal_priorizacao"]
    )
    return out


def priority_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize portfolio impact by prioritization bucket."""

    out = prepare_executive_df(df)
    summary = (
        out.groupby("sinal_priorizacao", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            unidades_vendidas=("qt_items_vendidos", "sum"),
            receita_liquida=("receita_liquida", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
            td_rate_medio=("td_rate", "mean"),
            priority_score_medio=("priority_score", "mean"),
        )
        .reset_index()
    )
    total_revenue = summary["receita_liquida"].sum()
    total_returned = summary["itens_retornados"].sum()
    summary["share_receita"] = np.where(
        total_revenue > 0, summary["receita_liquida"] / total_revenue, 0
    )
    summary["share_itens_retornados"] = np.where(
        total_returned > 0, summary["itens_retornados"] / total_returned, 0
    )
    return summary.sort_values("itens_retornados", ascending=False).reset_index(drop=True)


def top_offenders(
    df: pd.DataFrame,
    bucket: str | None = "Priorizar melhoria",
    n: int = 20,
    min_items_returned: int = 5,
) -> pd.DataFrame:
    """Return top offender products ranked by priority score and returned items."""

    out = prepare_executive_df(df)
    if bucket is not None:
        out = out[out["sinal_priorizacao"].eq(bucket)]
    out = out[out["qt_items_returned"].fillna(0) >= min_items_returned]
    return out.sort_values(
        ["priority_score", "qt_items_returned", "td_rate"],
        ascending=[False, False, False],
    ).head(n)


def category_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate product-level results by category for benchmarking."""

    out = prepare_executive_df(df)
    category = (
        out.groupby("category", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            unidades_vendidas=("qt_items_vendidos", "sum"),
            receita_liquida=("receita_liquida", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
            produtos_priorizar=(
                "sinal_priorizacao",
                lambda s: int((s == "Priorizar melhoria").sum()),
            ),
            td_rate_medio_produto=("td_rate", "mean"),
            td_rate_mediano_produto=("td_rate", "median"),
        )
        .reset_index()
    )
    category["td_rate_categoria_recalculado"] = np.where(
        category["unidades_vendidas"] > 0,
        category["itens_retornados"] / category["unidades_vendidas"],
        0,
    )
    return category.sort_values(
        ["produtos_priorizar", "itens_retornados"], ascending=False
    ).reset_index(drop=True)


def problema_tipo_split(df: pd.DataFrame) -> pd.DataFrame:
    """Breakdown T&D reversas by problem type (Físico / Logístico / Desistência).

    Uses the pre-aggregated qt_reversas_fisico and qt_reversas_logistico columns
    from the executive table.  The remainder (Outros + Desistência) is inferred
    as total reversas minus the two explicit buckets.
    """

    out = prepare_executive_df(df)
    total_reversas = float(out["qt_reversas"].fillna(0).sum())
    qt_fisico = (
        float(out["qt_reversas_fisico"].fillna(0).sum())
        if "qt_reversas_fisico" in out.columns
        else 0.0
    )
    qt_logistico = (
        float(out["qt_reversas_logistico"].fillna(0).sum())
        if "qt_reversas_logistico" in out.columns
        else 0.0
    )
    qt_outros = max(0.0, total_reversas - qt_fisico - qt_logistico)

    rows = [
        {"tipo_problema": "Físico", "qt_reversas": qt_fisico},
        {"tipo_problema": "Logístico", "qt_reversas": qt_logistico},
        {"tipo_problema": "Outros / Desistência", "qt_reversas": qt_outros},
    ]
    result = pd.DataFrame(rows)
    result["pct_total"] = result["qt_reversas"].div(
        total_reversas if total_reversas > 0 else 1.0
    )
    return result.sort_values("qt_reversas", ascending=False).reset_index(drop=True)


def tag_summary_from_executive(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize top problem tags already pivoted in the executive table."""

    out = prepare_executive_df(df)
    frames = []
    for rank in (1, 2, 3):
        problem_col = f"top_{rank}_problema"
        pct_col = f"top_{rank}_pct"
        if problem_col in out.columns:
            tmp = out[["product_name", "sinal_priorizacao", problem_col, pct_col]].copy()
            tmp = tmp.rename(columns={problem_col: "problema_tag", pct_col: "pct_no_produto"})
            tmp["rank"] = rank
            frames.append(tmp)
    if not frames:
        return pd.DataFrame(columns=["problema_tag", "produtos", "pct_medio_no_produto"])

    long_df = pd.concat(frames, ignore_index=True)
    long_df = long_df.dropna(subset=["problema_tag"])
    long_df = long_df[~long_df["problema_tag"].isin(TOP_PROBLEMAS_EXCLUDED_TAGS)]
    return (
        long_df.groupby("problema_tag", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            produtos_priorizar=(
                "sinal_priorizacao",
                lambda s: int((s == "Priorizar melhoria").sum()),
            ),
            pct_medio_no_produto=("pct_no_produto", "mean"),
        )
        .reset_index()
        .sort_values(["produtos_priorizar", "produtos"], ascending=False)
        .reset_index(drop=True)
    )


def get_tipo_de_acao(top_1_tag: str | None) -> str:
    """Map the top-1 return tag to a recommended action type for the PF team.

    Returns one of: Modelagem, Grade, Tecido, PDP/Comunicação,
    Investigar tags negativas, Logístico — fora do escopo PF,
    or Investigação adicional.
    """
    if top_1_tag is None:
        return "Investigação adicional"
    if top_1_tag in POSITIVE_TAGS:
        return "Investigar tags negativas"
    if top_1_tag in LOGISTICA_TAGS:
        return "Logístico — fora do escopo PF"
    return TAG_TO_ACTION.get(top_1_tag, "Investigação adicional")


def build_tweet_heuristic(row: Mapping[str, object]) -> str:
    """Generate a rule-based ≤80-word analyst tweet for one product.

    Input is a product-level row from the executive dataframe.
    Covers: top-3 problems, color/size concentration (if ≥50%), T&D rate
    vs category, top-3 leverage, trend (if not stable), and suggested action.
    When Top 1 tag is positive, emits a disclaimer to investigate negative tags.
    """

    def _f(v: object) -> float | None:
        try:
            return float(v) if v is not None else None
        except (TypeError, ValueError):
            return None

    top_1_tag = row.get("top_1_problema")
    tipo_de_acao = get_tipo_de_acao(top_1_tag if isinstance(top_1_tag, str) else None)
    is_positive = isinstance(top_1_tag, str) and top_1_tag in POSITIVE_TAGS

    parts: list[str] = []

    # Positive tags disclaimer — replaces the offenders section
    if is_positive:
        parts.append(
            "⚠️ Tags predominantes são positivas. "
            "Recomenda-se investigar as tags negativas para diagnóstico completo."
        )
    else:
        # Top 3 problems
        tags: list[str] = []
        for rank in (1, 2, 3):
            tag = row.get(f"top_{rank}_problema")
            pct = _f(row.get(f"top_{rank}_pct"))
            if tag:
                suffix = f" ({pct:.0%})" if pct is not None else ""
                tags.append(f"{tag}{suffix}")
        if tags:
            parts.append(f"Ofensores: {', '.join(tags)}.")

    # Color concentration (≥50%)
    cor = row.get("principal_cor_afetada")
    cor_pct = _f(row.get("principal_cor_pct"))
    if cor and cor_pct is not None and cor_pct >= 0.50:
        parts.append(f"Cor crítica: {cor} ({cor_pct:.0%} das reversas).")

    # Size concentration (≥50%)
    tam = row.get("principal_tamanho_afetado")
    tam_pct = _f(row.get("principal_tamanho_pct"))
    if tam and tam_pct is not None and tam_pct >= 0.50:
        parts.append(f"Tamanho crítico: {tam} ({tam_pct:.0%} das reversas).")

    # T&D rate vs category
    td_rate = _f(row.get("td_rate"))
    td_cat = _f(row.get("td_rate_categoria"))
    if td_rate is not None and td_cat is not None:
        delta = td_rate - td_cat
        direction = "acima" if delta >= 0 else "abaixo"
        parts.append(
            f"T&D: {td_rate:.1%} produto vs {td_cat:.1%} categoria "
            f"({abs(delta):.1%}pp {direction})."
        )

    # Top-3 leverage
    pct_top3 = _f(row.get("pct_top_3_total"))
    if pct_top3 is not None:
        parts.append(f"Top 3 concentra {pct_top3:.0%} das reversas tagueadas.")

    # Trend (skip stable / insufficient)
    trend = row.get("tendencia_reversas")
    if trend and trend not in ("Estável", "Sem volume para tendência"):
        parts.append(f"Tendência recente: {trend}.")

    # Suggested action type
    parts.append(f"Ação sugerida: {tipo_de_acao}.")

    result = " ".join(parts)
    words = result.split()
    if len(words) > 80:
        result = " ".join(words[:80]) + "…"
    return result or "Dados insuficientes para síntese."


def build_scorecard_product_view(
    executive_df: pd.DataFrame,
    scorecard_df: pd.DataFrame,
    n: int = 50,
    only_buckets: Sequence[str] | None = None,
) -> pd.DataFrame:
    """Merge the executive T&D table with the portfolio scorecard pillars.

    Parameters
    ----------
    executive_df:
        Output of prepare_executive_df(). Must contain product_name, category,
        gender, td_rate, td_rate_categoria, top_N_problema/pct columns.
    scorecard_df:
        Must contain: product_name, cluster, score_tracao_comercial,
        score_unit_economics, score_satisfacao_marca,
        td_produto_pct, td_categoria_pct.
    n:
        Max rows returned, sorted by priority_score DESC.
    only_buckets:
        If provided, filter to these sinal_priorizacao values before merging.
    """

    exec_out = prepare_executive_df(executive_df)
    if only_buckets is not None:
        exec_out = exec_out[exec_out["sinal_priorizacao"].isin(only_buckets)]

    merged = exec_out.merge(
        scorecard_df[
            [
                "product_name",
                "cluster",
                "score_tracao_comercial",
                "score_unit_economics",
                "score_satisfacao_marca",
                "td_produto_pct",
                "td_categoria_pct",
            ]
        ],
        on="product_name",
        how="left",
    )

    # Build "Top 3 motivos" string from pivoted columns already in executive_df
    def _top3_str(row: pd.Series) -> str:
        parts = []
        for rank in (1, 2, 3):
            tag = row.get(f"top_{rank}_problema")
            pct = row.get(f"top_{rank}_pct")
            if pd.isna(tag) or tag is None:
                continue
            pct_str = f"{pct:.0%}" if pd.notna(pct) else ""
            parts.append(f"{rank}. {tag} ({pct_str})")
        return "<br>".join(parts) if parts else "—"

    merged["top_3_motivos_td"] = merged.apply(_top3_str, axis=1)
    merged["tweet_analitico"] = merged.apply(build_tweet_heuristic, axis=1)

    cols = [
        "product_name",
        "category",
        "gender",
        "cluster",
        "sinal_priorizacao",
        "td_score",
        "score_tracao_comercial",
        "score_unit_economics",
        "score_satisfacao_marca",
        "td_categoria_pct",
        "td_produto_pct",
        "td_rate",
        "td_rate_categoria",
        "tendencia_reversas",
        "top_3_motivos_td",
        "tweet_analitico",
        "priority_score",
        "qt_items_returned",
        "receita_liquida",
    ]
    available = [c for c in cols if c in merged.columns]
    return (
        merged[available]
        .sort_values("priority_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


def _apply_workbook_formatting(wb: object) -> None:
    """Apply professional formatting to all sheets: header colors, tab colors,
    row coloring by sinal_priorizacao, freeze pane, and column widths."""
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    header_fill = PatternFill("solid", fgColor="1F497D")
    header_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    body_font = Font(name="Arial", size=10)

    for ws in wb.worksheets:  # type: ignore[attr-defined]
        name = ws.title

        # Tab color
        color = _TAB_COLORS.get(name)
        if color:
            ws.sheet_properties.tabColor = color

        # Freeze top row
        ws.freeze_panes = "A2"
        ws.row_dimensions[1].height = 28

        # Identify key columns
        sinal_col_idx: int | None = None
        mensagem_col_idx: int | None = None
        for cell in ws[1]:
            if cell.value == "sinal_priorizacao":
                sinal_col_idx = cell.column
            if cell.value == "mensagem_consolidada":
                mensagem_col_idx = cell.column

        # Header row
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        # Body: font + row coloring + alignment
        for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
            sinal_val = row[sinal_col_idx - 1].value if sinal_col_idx else None
            row_fill = None
            if sinal_val and str(sinal_val) in _SINAL_ROW_COLORS:
                row_fill = PatternFill("solid", fgColor=_SINAL_ROW_COLORS[str(sinal_val)])
            for cell in row:
                cell.font = body_font
                if row_fill:
                    cell.fill = row_fill
                if mensagem_col_idx and cell.column == mensagem_col_idx:
                    cell.alignment = Alignment(wrap_text=True, vertical="top")
                else:
                    cell.alignment = Alignment(vertical="center")
            if name == "tweets_analiticos":
                ws.row_dimensions[row_idx].height = 90

        # Column widths
        wide_cols = {"mensagem_consolidada", "tweet_analitico", "top_3_motivos_td", "comentarios_amostra"}
        medium_cols = {"product_name", "resumo_pre_llm"}
        for col in ws.columns:
            hdr = str(col[0].value or "")
            letter = get_column_letter(col[0].column)
            if hdr == "mensagem_consolidada":
                ws.column_dimensions[letter].width = 90
            elif hdr in wide_cols:
                ws.column_dimensions[letter].width = 60
            elif hdr in medium_cols:
                ws.column_dimensions[letter].width = 35
            else:
                max_len = max((len(str(c.value or "")) for c in col), default=8)
                ws.column_dimensions[letter].width = min(max(max_len + 2, 10), 32)


def export_priority_workbook(
    executive_df: pd.DataFrame,
    output_path: str,
    scorecard_view_df: pd.DataFrame | None = None,
) -> None:
    """Export core analysis tabs to an Excel workbook.

    Tab order: farol_executivo, resumo_farol, benchmark_categoria, resumo_tags,
    priorizar_melhoria, relatorio_pf (if provided).
    """

    executive = prepare_executive_df(executive_df)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        executive.to_excel(writer, sheet_name="farol_executivo", index=False)
        priority_summary(executive).to_excel(writer, sheet_name="resumo_farol", index=False)
        category_benchmark(executive).to_excel(writer, sheet_name="benchmark_categoria", index=False)
        tag_summary_from_executive(executive).to_excel(writer, sheet_name="resumo_tags", index=False)
        top_offenders(executive, bucket="Priorizar melhoria", n=50).to_excel(
            writer, sheet_name="priorizar_melhoria", index=False
        )
        if scorecard_view_df is not None:
            scorecard_view_df.to_excel(writer, sheet_name="relatorio_pf", index=False)
        _apply_workbook_formatting(writer.book)


## 6. Preparação e normalização dos DataFrames

Aplica validações, coerção de tipos e ordenação padrão sobre os dados brutos extraídos.

**Entrada:** `executive_df`, `df_scores_raw`
**Saída:** `executive_prepared_df`, `scorecard_df`
**Quando mexer:** ao adicionar novo campo de normalização ou alterar join do scorecard.


In [ ]:
# ── Preparação do DataFrame executivo ────────────────────────────────────────
# Valida colunas obrigatórias, coerce numéricos e ordena por priority_score DESC.
executive_prepared_df = prepare_executive_df(executive_df)

print(f"✅ executive_prepared_df: {executive_prepared_df.shape}")
display(executive_prepared_df.head(3))


In [ ]:
# ── Construção do scorecard_df ────────────────────────────────────────────────
# Adiciona td_produto_pct, td_categoria_pct (de executive_prepared_df) e
# cluster (de portfolio_cluster) ao scorecard de pilares.
# Nota: eval_produto_portfolio não tem cluster — ele vem da CTE portfolio_clustering via executive_df.

_td_ref = (
    executive_prepared_df[["product_name", "td_rate", "td_rate_categoria"]]
    .drop_duplicates("product_name")
    .rename(columns={"td_rate": "td_produto_pct", "td_rate_categoria": "td_categoria_pct"})
)

_cluster_ref = (
    executive_prepared_df[["product_name", "portfolio_cluster"]]
    .drop_duplicates("product_name")
    .rename(columns={"portfolio_cluster": "cluster"})
)

scorecard_df = (
    df_scores_raw
    .merge(_td_ref, on="product_name", how="left")
    .merge(_cluster_ref, on="product_name", how="left")
)

_coverage = scorecard_df["score_tracao_comercial"].notna().sum()
print(f"✅ scorecard_df: {scorecard_df.shape}")
print(f"   Scores populados: {_coverage}/{len(scorecard_df)} produto(s)")
print(f"   Produtos sem scorecard (ficarão com —): {len(scorecard_df) - _coverage}")


## 7. Classificação e score de priorização

Recomputa `sinal_priorizacao` em Python (espelho do SQL) para QA e adiciona `sinal_kill_keep`.

**Entrada:** `executive_prepared_df`
**Saída:** `classified_df` (com `sinal_priorizacao_python` e `sinal_confere_sql`),
  `executive_prepared_df` com coluna `sinal_kill_keep` adicionada
**Quando mexer:** ao ajustar thresholds (atenção: deve estar em sincronia com o SQL).


In [ ]:
# ── QA: Python vs SQL ────────────────────────────────────────────────────────
# classify_priority reimplementa a lógica SQL em Python para validação cruzada.
classified_df = classify_priority(executive_prepared_df)

match_rate = classified_df["sinal_confere_sql"].mean()
n_divergencias = (~classified_df["sinal_confere_sql"]).sum()

print(f"Match Python vs SQL: {match_rate:.2%}  ({n_divergencias} divergências)")

if n_divergencias > 0:
    divergencias = classified_df.loc[
        ~classified_df["sinal_confere_sql"],
        [
            "product_name",
            "sinal_priorizacao",
            "sinal_priorizacao_python",
            "priority_score",
            "td_score",
            "commercial_score_v2",
            "qt_items_vendidos",
            "qt_items_returned",
        ],
    ]
    print("\n⚠️  Divergências encontradas:")
    display(divergencias.head(50))


In [ ]:
# ── sinal_kill_keep ──────────────────────────────────────────────────────────
# Regra QA: Alta T&D + Alta Tração → Priorizar | Alta T&D + Baixa Tração → Avaliar saída.
# Fonte: lógica original de TD_Priorizacao_Melhorias_v20260611.ipynb (célula b402d21a).
# ⚠️ Não alterar thresholds (0.70 / 0.60 / 30 / 5) sem revisão metodológica.
executive_prepared_df["sinal_kill_keep"] = np.select(
    [
        (executive_prepared_df["qt_items_vendidos"] < 30)
        | (executive_prepared_df["qt_items_returned"] < 5),
        (executive_prepared_df["td_score"] >= 0.70)
        & (executive_prepared_df["commercial_score_v2"] >= 0.60),
        (executive_prepared_df["td_score"] >= 0.70)
        & (executive_prepared_df["commercial_score_v2"] < 0.60),
    ],
    [
        "Sem evidência suficiente",
        "Priorizar melhoria",
        "Não Priorizar (Avaliar Descontinuação/Reformulação)",
    ],
    default="Não priorizar agora",
)

print("✅ sinal_kill_keep adicionado ao executive_prepared_df")
print(executive_prepared_df["sinal_kill_keep"].value_counts().to_string())


## 8. Diagnósticos executivos

Tabelas intermediárias agregadas usadas para construir os outputs finais e revisar o comportamento do Farol.


In [ ]:
# ── §1. Resumo do portfólio por farol ────────────────────────────────────────
summary_df = priority_summary(executive_prepared_df)
print("summary_df:", summary_df.shape)
display(summary_df)


In [ ]:
# ── §2. Top produtos — Priorizar melhoria ────────────────────────────────────
priorizar_melhoria_raw_df = top_offenders(
    executive_prepared_df,
    bucket="Priorizar melhoria",
    n=50,
)
print("priorizar_melhoria_raw_df:", priorizar_melhoria_raw_df.shape)
display(priorizar_melhoria_raw_df[[
    "product_name", "category", "sinal_priorizacao",
    "priority_score", "td_score", "commercial_score_v2",
    "td_rate", "td_rate_categoria", "qt_items_returned",
]].head(10))


In [ ]:
# ── §3. Alerta em produto relevante ─────────────────────────────────────────
alerta_raw_df = top_offenders(
    executive_prepared_df,
    bucket="Alerta em produto relevante",
    n=20,
)
print("alerta_raw_df:", alerta_raw_df.shape)
display(alerta_raw_df[[
    "product_name", "category", "td_score", "commercial_score_v2",
    "td_rate", "receita_liquida",
]].head(10))


In [ ]:
# ── §4. Benchmark por categoria ──────────────────────────────────────────────
benchmark_categoria_raw_df = category_benchmark(executive_prepared_df)
print("benchmark_categoria_raw_df:", benchmark_categoria_raw_df.shape)
display(benchmark_categoria_raw_df)


In [ ]:
# ── §5. Top problemas por tag ────────────────────────────────────────────────
top_problemas_raw_df = tag_summary_from_executive(executive_prepared_df)
print("top_problemas_raw_df:", top_problemas_raw_df.shape)
display(top_problemas_raw_df.head(20))


In [ ]:
# ── §6. Split por tipo de problema ──────────────────────────────────────────
df_tipo = problema_tipo_split(executive_prepared_df)
print("df_tipo:", df_tipo.shape)
display(df_tipo)


In [ ]:
# ── §7. Tendência de reversas ────────────────────────────────────────────────
if "tendencia_reversas" in executive_prepared_df.columns:
    df_trend = (
        executive_prepared_df
        .groupby("tendencia_reversas", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            reversas_ultimos_3m=("reversas_ultimos_3m", "sum"),
            reversas_3m_anteriores=("reversas_3m_anteriores", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
        )
        .reset_index()
        .sort_values("itens_retornados", ascending=False)
        .reset_index(drop=True)
    )
    display(df_trend)
else:
    df_trend = None
    print("⚠️  Coluna tendencia_reversas não encontrada.")


In [ ]:
# ── §8. Concentração de cor e tamanho ────────────────────────────────────────
_top_buckets = ["Priorizar melhoria", "Alerta em produto relevante"]
df_prep_top = executive_prepared_df[
    executive_prepared_df["sinal_priorizacao"].isin(_top_buckets)
].copy()

_has_cor = (
    df_prep_top["principal_cor_pct"].fillna(0).ge(0.50)
    if "principal_cor_pct" in df_prep_top.columns
    else pd.Series(False, index=df_prep_top.index)
)
_has_tam = (
    df_prep_top["principal_tamanho_pct"].fillna(0).ge(0.50)
    if "principal_tamanho_pct" in df_prep_top.columns
    else pd.Series(False, index=df_prep_top.index)
)
df_conc = df_prep_top[_has_cor | _has_tam].copy()
print(f"Produtos com concentração ≥50% em cor ou tamanho: {len(df_conc)}")
if not df_conc.empty:
    cols_conc = [c for c in [
        "product_name", "category", "principal_cor_afetada", "principal_cor_pct",
        "principal_tamanho_afetado", "principal_tamanho_pct",
    ] if c in df_conc.columns]
    display(df_conc[cols_conc].head(20))


In [ ]:
# ── §9. Visão produto com scorecard (relatorio_pf) ───────────────────────────
scorecard_view_df = build_scorecard_product_view(
    executive_df=executive_prepared_df,
    scorecard_df=scorecard_df,
    n=50,
    only_buckets=["Priorizar melhoria", "Alerta em produto relevante", "Monitorar"],
)
print(f"scorecard_view_df: {scorecard_view_df.shape}")
display(scorecard_view_df.head(10))


## 9. QA e validações

Garante que os dados estão confiáveis **antes** de exportar ou atualizar a planilha.

**Entrada:** `executive_prepared_df`, `classified_df`
**Saída:** confirmação de QA ou exceção detalhada
**Quando mexer:** ao adicionar novo check ou ajustar tolerância de divergência.

> Este bloco deve ser executado antes do Bloco 11. Se falhar, **não prosseguir**.


In [ ]:
# ── Helpers de QA ────────────────────────────────────────────────────────────
def assert_not_empty(df: pd.DataFrame, df_name: str) -> None:
    "Levanta ValueError se o DataFrame estiver vazio."
    if df is not None and df.empty:
        raise ValueError(f"❌ QA FALHOU: {df_name} está vazio")

# ── QA 1: DataFrames não vazios ───────────────────────────────────────────────
assert_not_empty(executive_prepared_df, "executive_prepared_df")
assert_not_empty(df_scores_raw, "df_scores_raw")
assert_not_empty(summary_df, "summary_df")
print("✅ QA 1: DataFrames não vazios — OK")

# ── QA 2: Colunas obrigatórias ────────────────────────────────────────────────
validate_columns(executive_prepared_df, EXECUTIVE_REQUIRED_COLUMNS, "executive_prepared_df")
print("✅ QA 2: Colunas obrigatórias — OK")

# ── QA 3: Sinal Python vs SQL ─────────────────────────────────────────────────
if "sinal_confere_sql" in classified_df.columns:
    match_rate = classified_df["sinal_confere_sql"].mean()
    if match_rate < 1.0:
        n_div = (~classified_df["sinal_confere_sql"]).sum()
        raise ValueError(
            f"❌ QA FALHOU: Divergência Python vs SQL. "
            f"Match rate: {match_rate:.2%} ({n_div} produtos divergentes). "
            f"Verifique se os thresholds do SQL e do PrioritizationThresholds estão sincronizados."
        )
    print(f"✅ QA 3: Sinal Python vs SQL — {match_rate:.2%} match")
else:
    warnings.warn("classified_df sem coluna sinal_confere_sql — QA 3 não executado.")

# ── QA 4: Grain (sem duplicados por produto) ──────────────────────────────────
key_cols = ["product_name", "category", "gender"]
dup_products = executive_prepared_df.duplicated(key_cols).sum()
if dup_products > 0:
    raise ValueError(
        f"❌ QA FALHOU: {dup_products} linhas duplicadas em executive_prepared_df. "
        f"Grain esperado: product_name × category × gender."
    )
print(f"✅ QA 4: Grain sem duplicados — OK ({len(executive_prepared_df)} linhas)")

# ── QA 5: Ranges de scores [0, 1] ────────────────────────────────────────────
for col in ["priority_score", "td_score", "commercial_score_v2", "td_rate"]:
    if col in executive_prepared_df.columns:
        vals = executive_prepared_df[col].dropna()
        invalid = vals.lt(0).sum() + vals.gt(1).sum()
        if invalid > 0:
            raise ValueError(
                f"❌ QA FALHOU: Coluna '{col}' possui {invalid} valores fora de [0, 1]."
            )
print("✅ QA 5: Ranges de scores — OK")

print("\n✅ QA concluído com sucesso — pipeline pronto para construção dos payloads.")


## 10. Construção dos outputs finais

Gera os DataFrames finais no formato de consumo/exportação, com nomes de coluna legíveis.

**Entrada:** `executive_prepared_df`, `priorizar_melhoria_raw_df`, `scorecard_view_df`,
  `benchmark_categoria_raw_df`, `top_problemas_raw_df`, `summary_df`
**Saída:** `farol_completo_df`, `priorizar_melhoria_df`, `relatorio_pf_df`,
  `benchmark_categoria_df`, `top_problemas_df`, `resumo_payload`
**Quando mexer:** ao adicionar/renomear colunas nos outputs ou ajustar formatação de valores.

> Nomes bonitos de coluna **só entram nesta camada**. Não renomear colunas técnicas antes.


In [ ]:
# ── Headers das abas (contrato com a planilha existente) ─────────────────────
# Alterar aqui propaga para QA e escrita segura no Bloco 11.

FAROL_COMPLETO_HEADERS = [
    "Produto", "Categoria", "Gênero", "Cluster", "Sinal Farol",
    "Priority Score", "TD Score", "Comm. Score v2",
    "TD Rate", "TD Categ.", "Δ vs Cat.",
    "Top 1 Problema", "Top 2", "Top 3",
    "Tendência", "Itens Retorn.", "Receita Líq.", "Share T&D",
]

PRIORIZAR_MELHORIA_HEADERS = [
    "Rank", "Produto", "Categoria", "Gênero", "Cluster",
    "Priority Score", "TD Score", "Comm. Score v2",
    "TD Rate", "TD Categ.", "Δ vs Cat.",
    "Top Problema 1", "Top 2", "Top 3",
    "Tendência", "Itens Retorn.", "Receita Líq.",
]

RELATORIO_PF_HEADERS = [
    "Produto", "Categoria", "Gênero", "Cluster", "Sinal Kill/Keep",
    "TD Score", "Score Tração", "Score Unit Econ.", "Score Satisf.",
    "TD Categoria", "TD Produto", "Δ vs Cat.",
    "Tendência", "Top 3 Motivos de T&D", "Diagnóstico (Tweet)",
    "Priority Score", "Itens Retorn.", "Receita Líq.",
]

BENCHMARK_CATEGORIA_HEADERS = [
    "Categoria", "Qtd Produtos", "Unid. Vendidas", "Receita Líq.",
    "Itens Retornados", "Produtos Priorizar",
    "TD Rate Médio", "TD Rate Mediano", "TD Rate Categoria", "Urgência",
]

TOP_PROBLEMAS_HEADERS = [
    "Tag do Problema", "Produtos Afetados", "Produtos 'Priorizar'",
    "% Médio no Produto", "Rank Severidade", "Classificação",
]

LOVABLE_PRIORIZACAO_HEADERS = [
    "Produto",
    "Priorização",
    "Cluster",
    "Receita Média Mensal",
    "Diff do T&D da Categoria",
    "Top 5 Motivos de T&D",
    "Qtd de Itens Retornados",
    "Tendência de Taxa de Devolução",
]

# ── Legado (mantidos para compatibilidade com debug/apêndice) ─────────────────
FAROL_EXECUTIVO_COLS = [
    "product_name", "category", "gender", "portfolio_cluster",
    "sinal_priorizacao", "sinal_kill_keep", "priority_score", "td_score", "commercial_score_v2",
    "td_rate", "td_rate_categoria", "delta_vs_categoria",
    "qt_items_vendidos", "receita_liquida", "qt_items_returned",
    "top_1_problema", "top_1_pct", "top_2_problema", "top_2_pct",
    "top_3_problema", "top_3_pct", "pct_top_3_total",
    "principal_cor_afetada", "principal_cor_pct",
    "principal_tamanho_afetado", "principal_tamanho_pct",
    "tendencia_reversas", "reversas_ultimos_3m", "reversas_3m_anteriores",
    "share_receita_portfolio", "share_td_portfolio",
]

PRIORIZAR_MELHORIA_COLS = [
    "product_name", "category", "gender", "portfolio_cluster",
    "priority_score", "td_score", "commercial_score_v2",
    "td_rate", "td_rate_categoria", "delta_vs_categoria",
    "top_1_problema", "top_1_pct", "top_2_problema", "top_2_pct",
    "top_3_problema", "top_3_pct",
    "tendencia_reversas", "qt_items_returned", "receita_liquida",
]

RELATORIO_PF_COLS = [
    "product_name", "category", "gender", "cluster",
    "sinal_priorizacao",
    "td_score", "score_tracao_comercial", "score_unit_economics", "score_satisfacao_marca",
    "td_categoria_pct", "td_produto_pct", "td_rate", "td_rate_categoria",
    "tendencia_reversas", "top_3_motivos_td", "tweet_analitico",
    "priority_score", "qt_items_returned", "receita_liquida",
]

BENCHMARK_CATEGORIA_COLS = [
    "category", "produtos", "unidades_vendidas", "receita_liquida",
    "itens_retornados", "produtos_priorizar",
    "td_rate_medio_produto", "td_rate_mediano_produto", "td_rate_categoria_recalculado",
]

TOP_PROBLEMAS_COLS = [
    "problema_tag", "produtos", "produtos_priorizar",
    "pct_medio_no_produto",
]


# ── Funções de validação de contrato ─────────────────────────────────────────
def assert_required_columns(df: pd.DataFrame, required_columns: list, df_name: str) -> None:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} sem colunas obrigatórias: {missing}")


def assert_output_headers(df: pd.DataFrame, expected_headers: list, payload_name: str) -> None:

    actual = list(df.columns)

    if actual != expected_headers:print("✅ Headers de contrato e funções de assert definidos")

        raise ValueError(

            f"{payload_name} com headers incorretos.\n"

            f"Esperado: {expected_headers}\n"        )
            f"Atual   : {actual}"

In [ ]:
# ── Funções de construção de payload ─────────────────────────────────────────
# Cada função produz um DataFrame com as colunas finais da planilha existente.

def _select_available(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    "Seleciona colunas disponíveis no df. Mantido para debug/apêndice."
    available = [c for c in cols if c in df.columns]
    return df[available].copy()


# ── Mapeamentos técnico → header da planilha ─────────────────────────────────
FAROL_COMPLETO_COLUMNS = {
    "product_name": "Produto",
    "category": "Categoria",
    "gender": "Gênero",
    "portfolio_cluster": "Cluster",
    "sinal_priorizacao": "Sinal Farol",
    "priority_score": "Priority Score",
    "td_score": "TD Score",
    "commercial_score_v2": "Comm. Score v2",
    "td_rate": "TD Rate",
    "td_rate_categoria": "TD Categ.",
    "delta_vs_categoria": "Δ vs Cat.",
    "top_1_problema": "Top 1 Problema",
    "top_2_problema": "Top 2",
    "top_3_problema": "Top 3",
    "tendencia_reversas": "Tendência",
    "qt_items_returned": "Itens Retorn.",
    "receita_liquida": "Receita Líq.",
    "share_td_portfolio": "Share T&D",
}

PRIORIZAR_MELHORIA_COLUMNS = {
    "product_name": "Produto",
    "category": "Categoria",
    "gender": "Gênero",
    "portfolio_cluster": "Cluster",
    "priority_score": "Priority Score",
    "td_score": "TD Score",
    "commercial_score_v2": "Comm. Score v2",
    "td_rate": "TD Rate",
    "td_rate_categoria": "TD Categ.",
    "delta_vs_categoria": "Δ vs Cat.",
    "top_1_problema": "Top Problema 1",
    "top_2_problema": "Top 2",
    "top_3_problema": "Top 3",
    "tendencia_reversas": "Tendência",
    "qt_items_returned": "Itens Retorn.",
    "receita_liquida": "Receita Líq.",
}

RELATORIO_PF_COLUMNS = {
    "product_name": "Produto",
    "category": "Categoria",
    "gender": "Gênero",
    "cluster": "Cluster",
    "sinal_pf_final": "Sinal Kill/Keep",
    "td_score": "TD Score",
    "score_tracao_comercial": "Score Tração",
    "score_unit_economics": "Score Unit Econ.",
    "score_satisfacao_marca": "Score Satisf.",
    "td_categoria_pct": "TD Categoria",
    "td_produto_pct": "TD Produto",
    "delta_vs_categoria": "Δ vs Cat.",
    "tendencia_reversas": "Tendência",
    "top_3_motivos_td": "Top 3 Motivos de T&D",
    "tweet_analitico": "Diagnóstico (Tweet)",
    "priority_score": "Priority Score",
    "qt_items_returned": "Itens Retorn.",
    "receita_liquida": "Receita Líq.",
}


def build_farol_completo_payload(executive_df: pd.DataFrame) -> pd.DataFrame:
    df = executive_df.copy()
    if "portfolio_cluster" not in df.columns and "cluster" in df.columns:
        df["portfolio_cluster"] = df["cluster"]
    assert_required_columns(df, list(FAROL_COMPLETO_COLUMNS.keys()), "executive_df → Farol Completo")
    df = df.sort_values(
        ["priority_score", "qt_items_returned", "receita_liquida"],
        ascending=[False, False, False],
    )
    out = df[list(FAROL_COMPLETO_COLUMNS.keys())].rename(columns=FAROL_COMPLETO_COLUMNS)
    out = out[FAROL_COMPLETO_HEADERS]
    assert_output_headers(out, FAROL_COMPLETO_HEADERS, "farol_completo_df")
    return out.reset_index(drop=True)


def build_priorizar_melhoria_payload(executive_df: pd.DataFrame) -> pd.DataFrame:
    df = executive_df.copy()
    if "portfolio_cluster" not in df.columns and "cluster" in df.columns:
        df["portfolio_cluster"] = df["cluster"]
    assert_required_columns(
        df,
        ["sinal_priorizacao"] + list(PRIORIZAR_MELHORIA_COLUMNS.keys()),
        "executive_df → Priorizar Melhoria",
    )
    df = df[df["sinal_priorizacao"].eq("Priorizar melhoria")].copy()
    df = df.sort_values(
        ["priority_score", "qt_items_returned", "receita_liquida"],
        ascending=[False, False, False],
    )
    out = df[list(PRIORIZAR_MELHORIA_COLUMNS.keys())].rename(columns=PRIORIZAR_MELHORIA_COLUMNS)
    out.insert(0, "Rank", range(1, len(out) + 1))
    out = out[PRIORIZAR_MELHORIA_HEADERS]
    assert_output_headers(out, PRIORIZAR_MELHORIA_HEADERS, "priorizar_melhoria_df")
    return out.reset_index(drop=True)


def build_relatorio_pf_payload(scorecard_view_df: pd.DataFrame) -> pd.DataFrame:
    df = scorecard_view_df.copy()
    # Usa sinal_kill_keep quando disponível; fallback para sinal_priorizacao
    if "sinal_kill_keep" in df.columns:
        df["sinal_pf_final"] = df["sinal_kill_keep"]
    elif "sinal_priorizacao" in df.columns:
        df["sinal_pf_final"] = df["sinal_priorizacao"]
    else:
        df["sinal_pf_final"] = ""
    # Preenche colunas ausentes com NaN + aviso
    for col in RELATORIO_PF_COLUMNS.keys():
        if col not in df.columns:
            df[col] = float("nan")
            warnings.warn(f"Coluna ausente em scorecard_view_df, preenchida como NaN: {col}")
    df = df.sort_values(
        ["priority_score", "qt_items_returned", "receita_liquida"],
        ascending=[False, False, False],
    )
    out = df[list(RELATORIO_PF_COLUMNS.keys())].rename(columns=RELATORIO_PF_COLUMNS)
    out = out[RELATORIO_PF_HEADERS]
    assert_output_headers(out, RELATORIO_PF_HEADERS, "relatorio_pf_df")
    return out.reset_index(drop=True)


def build_benchmark_categoria_payload(benchmark_df: pd.DataFrame) -> pd.DataFrame:
    df = benchmark_df.copy()
    assert_required_columns(df, [
        "category", "produtos", "unidades_vendidas", "receita_liquida",
        "itens_retornados", "produtos_priorizar",
        "td_rate_medio_produto", "td_rate_mediano_produto", "td_rate_categoria_recalculado",
    ], "benchmark_categoria_raw_df")

    def _urgencia(row):
        n = row.get("produtos_priorizar", 0) or 0
        if n >= 3:
            return "🟠 Atenção"
        if n >= 1:
            return "🟡 Média"
        return "🟢 Baixa"

    out = pd.DataFrame({
        "Categoria":          df["category"].values,
        "Qtd Produtos":       df["produtos"].values,
        "Unid. Vendidas":     df["unidades_vendidas"].values,
        "Receita Líq.":       df["receita_liquida"].values,
        "Itens Retornados":   df["itens_retornados"].values,
        "Produtos Priorizar": df["produtos_priorizar"].values,
        "TD Rate Médio":      df["td_rate_medio_produto"].values,
        "TD Rate Mediano":    df["td_rate_mediano_produto"].values,
        "TD Rate Categoria":  df["td_rate_categoria_recalculado"].values,
        "Urgência":           df.apply(_urgencia, axis=1).values,
    })
    out = out.sort_values(
        ["Produtos Priorizar", "Itens Retornados", "Receita Líq."],
        ascending=[False, False, False],
    )
    out = out[BENCHMARK_CATEGORIA_HEADERS]
    assert_output_headers(out, BENCHMARK_CATEGORIA_HEADERS, "benchmark_categoria_df")
    return out.reset_index(drop=True)


def classify_problem_tag(tag) -> str:
    if pd.isna(tag):
        return "🔍 Outros"
    tag = str(tag)
    if tag in LOGISTICA_TAGS:
        return "🚚 Logístico — fora do escopo PF"
    action = TAG_TO_ACTION.get(tag)
    if action == "Modelagem":
        return "✂️ Modelagem"
    if action == "Grade":
        return "📏 Grade/Tamanho"
    if action == "Tecido":
        return "🧵 Matéria-Prima"
    if action == "PDP/Comunicação":
        return "🖼️ PDP/Comunicação"
    if action:
        return f"🔍 {action}"
    return "🔍 Outros"


def build_top_problemas_payload(top_problemas_raw_df: pd.DataFrame) -> pd.DataFrame:
    df = top_problemas_raw_df.copy()
    assert_required_columns(df, [
        "problema_tag", "produtos", "produtos_priorizar", "pct_medio_no_produto",
    ], "top_problemas_raw_df")
    df = df.sort_values(
        ["produtos_priorizar", "produtos", "pct_medio_no_produto"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    out = pd.DataFrame({
        "Tag do Problema":      df["problema_tag"].values,
        "Produtos Afetados":    df["produtos"].values,
        "Produtos 'Priorizar'": df["produtos_priorizar"].values,
        "% Médio no Produto":   df["pct_medio_no_produto"].values,
        "Rank Severidade":      range(1, len(df) + 1),
        "Classificação":        df["problema_tag"].apply(classify_problem_tag).values,
    })
    out = out[TOP_PROBLEMAS_HEADERS]
    assert_output_headers(out, TOP_PROBLEMAS_HEADERS, "top_problemas_df")
    return out


# ── Resumo executivo ──────────────────────────────────────────────────────────
BUCKETS = [
    "Priorizar melhoria",
    "Alerta em produto relevante",
    "Monitorar",
    "Não priorizar agora",
    "Sem evidência suficiente",
]


def _fmt_brl_compact(value: float) -> str:
    v = float(value or 0)
    if abs(v) >= 1_000_000:
        return f"R$ {v / 1_000_000:.1f}M"
    if abs(v) >= 1_000:
        return f"R$ {v / 1_000:.1f}k"
    return f"R$ {v:,.0f}"


def _fmt_int_br(value: float) -> str:
    return f"{int(value or 0):,}".replace(",", ".")


def build_resumo_executivo_payload(executive_df: pd.DataFrame) -> dict:
    df = executive_df.copy()
    assert_required_columns(
        df,
        ["product_name", "sinal_priorizacao", "receita_liquida", "qt_items_returned"],
        "executive_df → Resumo Executivo",
    )
    grouped = (
        df.groupby("sinal_priorizacao", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            receita_liquida=("receita_liquida", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
        )
        .reset_index()
    )
    gmap = grouped.set_index("sinal_priorizacao").to_dict(orient="index")
    bucket_cards = {}
    for bucket in BUCKETS:
        v = gmap.get(bucket, {"produtos": 0, "receita_liquida": 0, "itens_retornados": 0})
        receita = float(v.get("receita_liquida", 0) or 0)
        itens = float(v.get("itens_retornados", 0) or 0)
        bucket_cards[bucket] = {
            "produtos": int(v.get("produtos", 0) or 0),
            "receita_liquida": receita,
            "receita_liq_fmt": _fmt_brl_compact(receita),
            "itens_retornados": int(itens),
            "itens_retornados_fmt": _fmt_int_br(itens),
        }
    generated_at = RUN_DATE.strftime("%d/%m/%Y %H:%M")
    total_produtos = int(df["product_name"].nunique())
    return {
        "generated_at": generated_at,
        "total_produtos": total_produtos,
        "periodo_texto": "12 meses de dados (Troquecommerce)",
        "subtitle": (
            f"Análise de {total_produtos} produtos · "
            f"12 meses de dados (Troquecommerce) · "
            f"Gerado em {generated_at}"
        ),
        "bucket_cards": bucket_cards,
    }


print("✅ Funções de payload e resumo executivo definidas")


# ── lovable_priorizacao: árvore de decisão v0 ─────────────────────────────────
# Classificação 2×2: taxa_devolucao × concentração da tag principal.
# Thresholds definidos na cell de parâmetros (LOVABLE_THRESHOLD_*).

def _classify_lovable(taxa_devolucao: float, tag_principal_pct: float) -> str:
    """Classifica produto na árvore de decisão lovable_priorizacao.

    Retorna uma das 4 categorias mutuamente exclusivas:
    - 'Priorizar Melhorias': alta taxa devol + alta concentração tag
    - 'Monitorar': baixa taxa devol + alta concentração tag
    - 'Menos Prioritário': alta taxa devol + baixa concentração tag
    - 'Fora do Escopo': baixa taxa devol + baixa concentração tag
    """
    alta_taxa = (taxa_devolucao or 0) > LOVABLE_THRESHOLD_TAXA_DEVOLUCAO
    alta_tag  = (tag_principal_pct or 0) > LOVABLE_THRESHOLD_TAG_CONCENTRACAO

    if alta_taxa and alta_tag:
        return "Priorizar Melhorias"
    if not alta_taxa and alta_tag:
        return "Monitorar"
    if alta_taxa and not alta_tag:
        return "Menos Prioritário"
    return "Fora do Escopo"


def _build_top5_motivos(row: pd.Series) -> str:
    """Concatena top 5 tags de problema de produto em uma string legível.

    Formato: 'tag1 (X%), tag2 (Y%), ...'
    Pula tags None/NaN.
    """
    parts = []
    for i in range(1, 6):
        tag_col = f"top_{i}_problema"
        pct_col = f"top_{i}_pct"
        tag = row.get(tag_col)
        pct = row.get(pct_col)
        if pd.notna(tag) and tag:
            pct_str = f" ({pct:.0%})" if pd.notna(pct) else ""
            parts.append(f"{tag}{pct_str}")
    return ", ".join(parts) if parts else ""


def build_lovable_priorizacao_payload(executive_df: pd.DataFrame) -> pd.DataFrame:
    """Constrói o DataFrame para a aba lovable_priorizacao.

    Lógica:
    1. Calcula taxa_devolucao = qt_devolucoes / qt_items_vendidos (apenas devoluções).
    2. Usa top_1_pct como concentração da tag principal.
    3. Aplica árvore de decisão 2×2 para classificar.
    4. Monta colunas complementares para priorização qualitativa.
    """
    df = executive_df.copy()

    # ── Validação de colunas obrigatórias ────────────────────────────────────
    required = [
        "product_name", "qt_devolucoes", "qt_items_vendidos",
        "portfolio_cluster", "receita_liquida", "delta_vs_categoria",
        "top_1_pct", "top_1_problema", "qt_items_returned",
        "tendencia_reversas",
    ]
    assert_required_columns(df, required, "executive_df → lovable_priorizacao")

    # ── Taxa de devolução (apenas devoluções, não trocas) ──────────────────
    df["taxa_devolucao"] = df["qt_devolucoes"] / df["qt_items_vendidos"].replace(0, float("nan"))

    # ── Classificação via árvore de decisão ────────────────────────────
    df["priorizacao_lovable"] = df.apply(
        lambda r: _classify_lovable(r["taxa_devolucao"], r["top_1_pct"]),
        axis=1,
    )

    # ── Receita média mensal (janela de 12 meses) ──────────────────────
    df["receita_media_mensal"] = df["receita_liquida"] / 12

    # ── Top 5 motivos concatenados ─────────────────────────────────
    df["top_5_motivos_td"] = df.apply(_build_top5_motivos, axis=1)

    # ── Ordenação: Priorizar Melhorias primeiro, depois por receita DESC ───
    ordem_priorizacao = {
        "Priorizar Melhorias": 1,
        "Monitorar": 2,
        "Menos Prioritário": 3,
        "Fora do Escopo": 4,
    }
    df["_sort_key"] = df["priorizacao_lovable"].map(ordem_priorizacao).fillna(99)
    df = df.sort_values(
        ["_sort_key", "receita_media_mensal", "qt_items_returned"],
        ascending=[True, False, False],
    )

    # ── Fallback: portfolio_cluster pode estar como 'cluster' ──────────────
    cluster_col = "portfolio_cluster" if "portfolio_cluster" in df.columns else "cluster"

    # ── Montagem do output ─────────────────────────────────────────
    out = pd.DataFrame({
        "Produto":                       df["product_name"].values,
        "Priorização":                   df["priorizacao_lovable"].values,
        "Cluster":                       df[cluster_col].values,
        "Receita Média Mensal":          df["receita_media_mensal"].values,
        "Diff do T&D da Categoria":      df["delta_vs_categoria"].values,
        "Top 5 Motivos de T&D":          df["top_5_motivos_td"].values,
        "Qtd de Itens Retornados":       df["qt_items_returned"].values,
        "Tendência de Taxa de Devolução": df["tendencia_reversas"].values,
    })

    out = out[LOVABLE_PRIORIZACAO_HEADERS]
    assert_output_headers(out, LOVABLE_PRIORIZACAO_HEADERS, "lovable_priorizacao_df")
    return out.reset_index(drop=True)


print("✅ Funções lovable_priorizacao definidas")


In [ ]:
# ── Construção dos payloads finais ────────────────────────────────────────────
farol_completo_df = build_farol_completo_payload(executive_prepared_df)

priorizar_melhoria_df = build_priorizar_melhoria_payload(executive_prepared_df)

if scorecard_view_df is not None and not scorecard_view_df.empty:
    relatorio_pf_df = build_relatorio_pf_payload(scorecard_view_df)
else:
    warnings.warn("scorecard_view_df não disponível — relatorio_pf_df será None.")
    relatorio_pf_df = None

benchmark_categoria_df = build_benchmark_categoria_payload(benchmark_categoria_raw_df)

top_problemas_df = build_top_problemas_payload(top_problemas_raw_df)

resumo_payload = build_resumo_executivo_payload(executive_prepared_df)

lovable_priorizacao_df = build_lovable_priorizacao_payload(executive_prepared_df)

print("✅ Payloads finais construídos")
print(f"  farol_completo_df      : {farol_completo_df.shape}")
print(f"  priorizar_melhoria_df  : {priorizar_melhoria_df.shape}")
print(f"  relatorio_pf_df        : {relatorio_pf_df.shape if relatorio_pf_df is not None else 'None'}")
print(f"  benchmark_categoria_df : {benchmark_categoria_df.shape}")
print(f"  top_problemas_df       : {top_problemas_df.shape}")
print(f"  resumo_payload buckets : {list(resumo_payload['bucket_cards'].keys())}")
print(f"  lovable_priorizacao_df : {lovable_priorizacao_df.shape}")

# ── QA dos payloads ───────────────────────────────────────────────────────────
assert_output_headers(farol_completo_df, FAROL_COMPLETO_HEADERS, "farol_completo_df")
assert_output_headers(priorizar_melhoria_df, PRIORIZAR_MELHORIA_HEADERS, "priorizar_melhoria_df")
assert_output_headers(benchmark_categoria_df, BENCHMARK_CATEGORIA_HEADERS, "benchmark_categoria_df")
assert_output_headers(top_problemas_df, TOP_PROBLEMAS_HEADERS, "top_problemas_df")
assert_output_headers(lovable_priorizacao_df, LOVABLE_PRIORIZACAO_HEADERS, "lovable_priorizacao_df")
if relatorio_pf_df is not None:
    assert_output_headers(relatorio_pf_df, RELATORIO_PF_HEADERS, "relatorio_pf_df")

if farol_completo_df.empty:
    raise ValueError("farol_completo_df vazio. Abortando — pipeline sem dados.")

if priorizar_melhoria_df.empty:
    warnings.warn("priorizar_melhoria_df vazio. A aba será limpa e ficará sem linhas de dados.")

if relatorio_pf_df is None or relatorio_pf_df.empty:
    warnings.warn("relatorio_pf_df vazio/None. Aba Relatório PF não será atualizada.")

print("✅ QA de payloads concluído")


## 11. Atualização/exportação dos resultados

Atualiza a planilha Google Sheets **existente** e (opcionalmente) exporta Excel de debug.

**Entrada:** todos os DataFrames de payload do Bloco 10
**Saída:** planilha atualizada / arquivo Excel de debug
**Quando mexer:** ao alterar autenticação, ID da planilha ou tabs exportadas.

> ⚠️ **Não criar uma nova planilha** — sempre usar `open_by_key(SPREADSHEET_ID)`.
>
> **Variáveis de ambiente necessárias:**
> - `GOOGLE_SERVICE_ACCOUNT_JSON` (Deepnote/CI) — JSON do service account com acesso ao Drive/Sheets
> - `TD_PRIORIZACAO_SPREADSHEET_ID` (opcional) — substitui o ID hardcoded no Bloco 2
> - `BQ_PROJECT_ID` (opcional) — substitui o projeto BigQuery padrão
>
> **Fallback local (Mac):** se `GOOGLE_SERVICE_ACCOUNT_JSON` não estiver definido,
> usa `gcloud auth print-access-token` (requer `gcloud auth login --enable-gdrive-access`).


In [ ]:
# ── Constantes de formatação para Sheets API v4 ───────────────────────────────
# Cores como dicts RGB (0.0–1.0), formato exigido pela Sheets API.
# Referência: documento de formatação TD_Priorizacao_Melhorias.

def hex_to_rgb(h: str) -> dict:
    """Converte hex 6 dígitos sem '#' para dict Sheets API RGB (0.0–1.0)."""
    h = h.lstrip("#")
    return {
        "red":   int(h[0:2], 16) / 255,
        "green": int(h[2:4], 16) / 255,
        "blue":  int(h[4:6], 16) / 255,
    }

# ── Cores base ────────────────────────────────────────────────────────────────
_WHITE       = hex_to_rgb("FFFFFF")
_GRAY_LIGHT  = hex_to_rgb("F5F5F5")  # linhas ímpares (0-indexed)
_GRAY_BORDER = hex_to_rgb("E0E0E0")  # bordas thin
_DARK_TEXT   = hex_to_rgb("2D2D2D")  # texto padrão de valor

# ── Sinal Farol / Kill-Keep ───────────────────────────────────────────────────
_SINAL_FMT = {
    "Priorizar melhoria":          {"bg": hex_to_rgb("FF4444"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Alerta em produto relevante": {"bg": hex_to_rgb("FFC107"), "fg": hex_to_rgb("1A1A1A"), "bold": True},
    "Monitorar":                   {"bg": hex_to_rgb("FF8C00"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Não priorizar agora":         {"bg": hex_to_rgb("4CAF50"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Sem evidência suficiente":    {"bg": hex_to_rgb("BDBDBD"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    # lovable_priorizacao categories
    "Priorizar Melhorias":           {"bg": hex_to_rgb("FF4444"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Menos Prioritário":             {"bg": hex_to_rgb("FFC107"), "fg": hex_to_rgb("1A1A1A"), "bold": True},
    "Fora do Escopo":                {"bg": hex_to_rgb("BDBDBD"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
}

# ── Cluster (somente cor de fonte; fundo da linha permanece alternado) ────────
_CLUSTER_FG = {
    "HERO":       hex_to_rgb("1565C0"),  # azul escuro
    "CORE":       hex_to_rgb("2E7D32"),  # verde escuro
    "LONG TAIL":  hex_to_rgb("6A1B9A"),  # roxo
    "KILL":       hex_to_rgb("B71C1C"),  # vermelho escuro (+ italic)
    "LANCAMENTO": hex_to_rgb("F57F17"),  # laranja
}

# ── Tendência (fundo da célula inteira) ───────────────────────────────────────
_TENDENCIA_BG = {
    "Aumentou nos últimos 3 meses":  hex_to_rgb("FFEBEE"),
    "Apareceu nos últimos 3 meses":  hex_to_rgb("FFEBEE"),
    "Estável":                        hex_to_rgb("F5F5F5"),
    "Caiu nos últimos 3 meses":       hex_to_rgb("E8F5E9"),
    "Sem volume para tendência":      hex_to_rgb("F5F5F5"),
}

# ── Alertas de threshold (Benchmark Categoria e Top Problemas) ────────────────
_THRESHOLD_FMT = {
    "critical": {"fg": hex_to_rgb("C62828"), "bold": True},
    "warning":  {"fg": hex_to_rgb("E65100"), "bold": True},
    "normal":   {"fg": _DARK_TEXT,            "bold": False},
}

# ── Configuração de formatação por aba ────────────────────────────────────────
# Índices de coluna são 0-based (A=0, B=1, ...).
# "sinal_col"    : índice da coluna com valor de sinal (para colorir fundo/texto)
# "cluster_col"  : índice da coluna Cluster (para colorir fonte)
# "tendencia_col": índice da coluna Tendência (para colorir fundo)
# "produto_col"  : índice da coluna Produto/nome (bold + left-align)
# "rank_col"     : índice da coluna Rank, se existir (bold vermelho)
# "number_formats": {col_idx: format_string}  — formato numérico por coluna
# "wrap_cols"    : colunas com wrap_text=True
# "threshold_col": coluna com alerta de threshold + "threshold_fn" classificadora
# "n_cols"       : número total de colunas da aba

FORMATTING_CONFIG = {
    "farol_completo": {
        "produto_col":    0,    # Produto
        "cluster_col":    3,    # Cluster
        "sinal_col":      4,    # Sinal Farol
        "tendencia_col":  14,   # Tendência
        "rank_col":       None,
        "threshold_col":  None,
        "n_cols":         18,
        "number_formats": {
            5:  "0.000",      # Priority Score
            6:  "0.000",      # TD Score
            7:  "0.000",      # Comm. Score v2
            8:  "0.0%",       # TD Rate
            9:  "0.0%",       # TD Categ.
            10: "0.0%",       # Δ vs Cat.
            15: "#,##0",      # Itens Retorn.
            16: "R$ #,##0",   # Receita Líq.
            17: "0.0%",       # Share T&D
        },
        "wrap_cols": [],
    },
    "priorizar_melhoria": {
        "produto_col":    1,    # Produto (col A = Rank)
        "cluster_col":    4,    # Cluster
        "sinal_col":      None, # toda aba é "Priorizar melhoria" — sem coluna sinal
        "tendencia_col":  14,   # Tendência
        "rank_col":       0,    # Rank
        "threshold_col":  None,
        "n_cols":         17,
        "number_formats": {
            5:  "0.000",      # Priority Score
            6:  "0.000",      # TD Score
            7:  "0.000",      # Comm. Score v2
            8:  "0.0%",       # TD Rate
            9:  "0.0%",       # TD Categ.
            10: "0.0%",       # Δ vs Cat.
            15: "#,##0",      # Itens Retorn.
            16: "R$ #,##0",   # Receita Líq.
        },
        "wrap_cols": [],
    },
    "relatorio_pf": {
        "produto_col":    0,    # Produto
        "cluster_col":    3,    # Cluster
        "sinal_col":      4,    # Sinal Kill/Keep
        "tendencia_col":  12,   # Tendência
        "rank_col":       None,
        "threshold_col":  None,
        "n_cols":         18,
        "number_formats": {
            5:  "0.0",        # TD Score
            6:  "0.0",        # Score Tração
            7:  "0.0",        # Score Unit Econ.
            8:  "0.0",        # Score Satisf.
            9:  "0.0%",       # TD Categoria
            10: "0.0%",       # TD Produto
            11: "0.0%",       # Δ vs Cat.
            15: "0.000",      # Priority Score
            16: "#,##0",      # Itens Retorn.
            17: "R$ #,##0",   # Receita Líq.
        },
        "wrap_cols": [13, 14],  # Top 3 Motivos (13) e Diagnóstico (14)
    },
    "benchmark_categoria": {
        "produto_col":    0,    # Categoria (bold left)
        "cluster_col":    None,
        "sinal_col":      None,
        "tendencia_col":  None,
        "rank_col":       None,
        "threshold_col":  5,    # Produtos Priorizar
        "threshold_fn":   lambda v: (
            "critical" if (v or 0) >= 2 else
            "warning"  if (v or 0) == 1 else
            "normal"
        ),
        "n_cols": 10,
        "number_formats": {
            1: "#,##0",       # Qtd Produtos
            2: "#,##0",       # Unid. Vendidas
            3: "R$ #,##0",    # Receita Líq.
            4: "#,##0",       # Itens Retornados
            5: "#,##0",       # Produtos Priorizar
            6: "0.0%",        # TD Rate Médio
            7: "0.0%",        # TD Rate Mediano
            8: "0.0%",        # TD Rate Categoria
        },
        "wrap_cols": [],
    },
    "top_problemas": {
        "produto_col":    0,    # Tag do Problema
        "cluster_col":    None,
        "sinal_col":      None,
        "tendencia_col":  None,
        "rank_col":       None,
        "threshold_col":  2,    # Produtos 'Priorizar'
        "threshold_fn":   lambda v: (
            "critical" if (v or 0) >= 5 else
            "warning"  if (v or 0) >= 2 else
            "normal"
        ),
        "n_cols": 6,
        "number_formats": {
            1: "#,##0",       # Produtos Afetados
            2: "#,##0",       # Produtos 'Priorizar'
            3: "0.0%",        # % Médio no Produto
            4: "#,##0",       # Rank Severidade
        },
        "wrap_cols": [],
    },
    "lovable_priorizacao": {
        "produto_col":    0,    # Produto
        "cluster_col":    2,    # Cluster
        "sinal_col":      1,    # Priorização
        "tendencia_col":  7,    # Tendência de Taxa de Devolução
        "rank_col":       None,
        "threshold_col":  None,
        "n_cols":         8,
        "number_formats": {
            3: "R$ #,##0",    # Receita Média Mensal
            4: "0.0%",        # Diff do T&D da Categoria
            6: "#,##0",       # Qtd de Itens Retornados
        },
        "wrap_cols": [5],     # Top 5 Motivos de T&D
    },
}

print("✅ Constantes de formatação Sheets API definidas")
print("✅ FORMATTING_CONFIG definido")


In [ ]:
# ── Helpers internos de construção de requests ────────────────────────────────

def _range(sheet_id: int, r0: int, r1: int, c0: int, c1: int) -> dict:
    """Range object Sheets API. r0/c0 inclusive, r1/c1 exclusive (0-indexed)."""
    return {
        "sheetId":          sheet_id,
        "startRowIndex":    r0,
        "endRowIndex":      r1,
        "startColumnIndex": c0,
        "endColumnIndex":   c1,
    }

def _cell_fmt(sheet_id: int, row_0: int, col_0: int, fmt: dict, fields: str) -> dict:
    """repeatCell request para UMA célula. row_0/col_0 são 0-indexed."""
    return {
        "repeatCell": {
            "range": _range(sheet_id, row_0, row_0 + 1, col_0, col_0 + 1),
            "cell":  {"userEnteredFormat": fmt},
            "fields": fields,
        }
    }

def _col_fmt(sheet_id: int, r0: int, r1: int, col_0: int, fmt: dict, fields: str) -> dict:
    """repeatCell request para uma coluna inteira no range r0..r1 (exclusive)."""
    return {
        "repeatCell": {
            "range": _range(sheet_id, r0, r1, col_0, col_0 + 1),
            "cell":  {"userEnteredFormat": fmt},
            "fields": fields,
        }
    }

def _row_fmt(sheet_id: int, row_0: int, n_cols: int, fmt: dict, fields: str) -> dict:
    """repeatCell request para uma linha inteira (0..n_cols)."""
    return {
        "repeatCell": {
            "range": _range(sheet_id, row_0, row_0 + 1, 0, n_cols),
            "cell":  {"userEnteredFormat": fmt},
            "fields": fields,
        }
    }


def build_base_requests(sheet_id: int, start_row: int, n_rows: int, n_cols: int) -> list:
    """
    Retorna requests para:
      1. Fundo alternado (branco/cinza) por linha de dado.
      2. Bordas thin cinza em toda a range de dados.
      3. Alinhamento central (horizontal=CENTER, vertical=MIDDLE) em toda a range.

    start_row : linha Excel 1-indexed (ex: 3).
    n_rows    : número de linhas de dado.
    n_cols    : número de colunas da aba.
    """
    reqs   = []
    start_0 = start_row - 1  # 0-indexed

    # ── 1. Fundo alternado ─────────────────────────────────────────────────
    # i=0 (primeira linha de dado) → branco; i=1 → cinza; etc.
    for i in range(n_rows):
        bg = _WHITE if i % 2 == 0 else _GRAY_LIGHT
        reqs.append(_row_fmt(
            sheet_id,
            start_0 + i,
            n_cols,
            {"backgroundColor": bg},
            "userEnteredFormat.backgroundColor",
        ))

    # ── 2. Bordas thin em toda a range ────────────────────────────────────
    thin = {"style": "SOLID", "colorStyle": {"rgbColor": _GRAY_BORDER}, "width": 1}
    reqs.append({
        "updateBorders": {
            "range":           _range(sheet_id, start_0, start_0 + n_rows, 0, n_cols),
            "top":             thin,
            "bottom":          thin,
            "left":            thin,
            "right":           thin,
            "innerHorizontal": thin,
            "innerVertical":   thin,
        }
    })

    # ── 3. Alinhamento central padrão ──────────────────────────────────────
    reqs.append({
        "repeatCell": {
            "range": _range(sheet_id, start_0, start_0 + n_rows, 0, n_cols),
            "cell": {
                "userEnteredFormat": {
                    "horizontalAlignment": "CENTER",
                    "verticalAlignment":   "MIDDLE",
                }
            },
            "fields": (
                "userEnteredFormat.horizontalAlignment,"
                "userEnteredFormat.verticalAlignment"
            ),
        }
    })

    return reqs


def build_column_format_requests(
    sheet_id: int,
    start_row: int,
    n_rows: int,
    config: dict,
) -> list:
    """
    Retorna requests para:
      1. Formato numérico por coluna (number_formats do config).
      2. Coluna Produto: bold + left-align + padding.
      3. Coluna Rank (se existir): bold vermelho C62828 size 11.
      4. Colunas com wrap_text (wrapStrategy=WRAP, verticalAlignment=TOP).
         - Coluna 14 em relatorio_pf (Diagnóstico): também italic + cinza 424242.
    """
    reqs    = []
    start_0 = start_row - 1
    end_0   = start_0 + n_rows

    # ── 1. Formatos numéricos ─────────────────────────────────────────────
    for col_0, pattern in config.get("number_formats", {}).items():
        reqs.append(_col_fmt(
            sheet_id, start_0, end_0, col_0,
            {"numberFormat": {"type": "NUMBER", "pattern": pattern}},
            "userEnteredFormat.numberFormat",
        ))

    # ── 2. Coluna Produto: bold + left-align ──────────────────────────────
    pc = config["produto_col"]
    reqs.append(_col_fmt(
        sheet_id, start_0, end_0, pc,
        {
            "textFormat":          {"bold": True},
            "horizontalAlignment": "LEFT",
        },
        "userEnteredFormat.textFormat.bold,userEnteredFormat.horizontalAlignment",
    ))

    # ── 3. Coluna Rank: bold vermelho C62828 size 11 ──────────────────────
    rc = config.get("rank_col")
    if rc is not None:
        reqs.append(_col_fmt(
            sheet_id, start_0, end_0, rc,
            {
                "textFormat": {
                    "bold":     True,
                    "fontSize": 11,
                    "foregroundColorStyle": {"rgbColor": hex_to_rgb("C62828")},
                },
                "horizontalAlignment": "CENTER",
            },
            (
                "userEnteredFormat.textFormat.bold,"
                "userEnteredFormat.textFormat.fontSize,"
                "userEnteredFormat.textFormat.foregroundColorStyle,"
                "userEnteredFormat.horizontalAlignment"
            ),
        ))

    # ── 4. Colunas com wrap_text ───────────────────────────────────────────
    for col_0 in config.get("wrap_cols", []):
        is_diagnostico = (col_0 == 14)
        fmt = {
            "wrapStrategy":        "WRAP",
            "verticalAlignment":   "TOP",
            "horizontalAlignment": "LEFT",
        }
        fields = (
            "userEnteredFormat.wrapStrategy,"
            "userEnteredFormat.verticalAlignment,"
            "userEnteredFormat.horizontalAlignment"
        )
        if is_diagnostico:
            fmt["textFormat"] = {
                "italic": True,
                "foregroundColorStyle": {"rgbColor": hex_to_rgb("424242")},
            }
            fields += (
                ",userEnteredFormat.textFormat.italic"
                ",userEnteredFormat.textFormat.foregroundColorStyle"
            )
        reqs.append(_col_fmt(sheet_id, start_0, end_0, col_0, fmt, fields))

    return reqs


def build_conditional_cell_requests(
    sheet_id: int,
    start_row: int,
    df: "pd.DataFrame",
    config: dict,
) -> list:
    """
    Retorna requests por célula com base no valor do DataFrame:
      - sinal_col:     fundo + texto colorido (sinal farol / kill-keep)
      - cluster_col:   cor da fonte (+ italic se KILL)
      - tendencia_col: fundo da célula
      - threshold_col: cor da fonte por threshold_fn

    Usa acesso posicional (list(row)) para robustez com nomes de coluna
    contendo caracteres especiais (Δ, acentos, emojis).
    """
    if len(df.columns) != config["n_cols"]:
        raise ValueError(
            f"build_conditional_cell_requests: DataFrame tem {len(df.columns)} colunas, "
            f"FORMATTING_CONFIG espera {config['n_cols']}.\n"
            f"Colunas do df: {list(df.columns)}"
        )

    reqs          = []
    start_0       = start_row - 1
    sinal_col     = config.get("sinal_col")
    cluster_col   = config.get("cluster_col")
    tendencia_col = config.get("tendencia_col")
    threshold_col = config.get("threshold_col")
    threshold_fn  = config.get("threshold_fn")

    for i, row in enumerate(df.itertuples(index=False)):
        row_0  = start_0 + i
        row_v  = list(row)  # acesso posicional — seguro com qualquer nome de coluna

        # ── Sinal Farol / Kill-Keep ────────────────────────────────────────
        if sinal_col is not None:
            sinal  = str(row_v[sinal_col] or "")
            fmt_s  = _SINAL_FMT.get(sinal)
            if fmt_s:
                reqs.append(_cell_fmt(
                    sheet_id, row_0, sinal_col,
                    {
                        "backgroundColor": fmt_s["bg"],
                        "textFormat": {
                            "bold":  fmt_s["bold"],
                            "foregroundColorStyle": {"rgbColor": fmt_s["fg"]},
                        },
                        "horizontalAlignment": "CENTER",
                    },
                    (
                        "userEnteredFormat.backgroundColor,"
                        "userEnteredFormat.textFormat.bold,"
                        "userEnteredFormat.textFormat.foregroundColorStyle,"
                        "userEnteredFormat.horizontalAlignment"
                    ),
                ))

        # ── Cluster: cor de fonte (+ italic se KILL) ───────────────────────
        if cluster_col is not None:
            cluster = str(row_v[cluster_col] or "").upper().strip()
            fg      = _CLUSTER_FG.get(cluster)
            if fg:
                reqs.append(_cell_fmt(
                    sheet_id, row_0, cluster_col,
                    {
                        "textFormat": {
                            "bold":   True,
                            "italic": (cluster == "KILL"),
                            "foregroundColorStyle": {"rgbColor": fg},
                        }
                    },
                    (
                        "userEnteredFormat.textFormat.bold,"
                        "userEnteredFormat.textFormat.italic,"
                        "userEnteredFormat.textFormat.foregroundColorStyle"
                    ),
                ))

        # ── Tendência: fundo da célula ─────────────────────────────────────
        if tendencia_col is not None:
            tend = str(row_v[tendencia_col] or "")
            bg   = _TENDENCIA_BG.get(tend)
            if bg:
                reqs.append(_cell_fmt(
                    sheet_id, row_0, tendencia_col,
                    {"backgroundColor": bg},
                    "userEnteredFormat.backgroundColor",
                ))

        # ── Threshold (Benchmark Categoria, Top Problemas) ─────────────────
        if threshold_col is not None and threshold_fn is not None:
            raw_val = row_v[threshold_col]
            level   = threshold_fn(raw_val)
            fmt_t   = _THRESHOLD_FMT.get(level, _THRESHOLD_FMT["normal"])
            reqs.append(_cell_fmt(
                sheet_id, row_0, threshold_col,
                {
                    "textFormat": {
                        "bold": fmt_t["bold"],
                        "foregroundColorStyle": {"rgbColor": fmt_t["fg"]},
                    }
                },
                (
                    "userEnteredFormat.textFormat.bold,"
                    "userEnteredFormat.textFormat.foregroundColorStyle"
                ),
            ))

    return reqs


def apply_tab_formatting(
    spreadsheet,
    worksheet,
    df: "pd.DataFrame",
    tab_key: str,
    start_row: int = 3,
) -> None:
    """
    Aplica formatação completa às linhas de dados recém-escritas via Sheets API batchUpdate.

    Garantias:
      - Nunca toca nas linhas 1 (banner) ou 2 (headers).
      - Envia todos os requests em UM único batchUpdate.
      - Não aplica formatação fora do range de dados (start_row .. start_row+len(df)-1).

    Parâmetros:
      spreadsheet : objeto gspread.Spreadsheet
      worksheet   : objeto gspread.Worksheet (destino)
      df          : DataFrame escrito (sem header, sem index)
      tab_key     : chave em FORMATTING_CONFIG
      start_row   : linha Excel onde os dados começam (default 3, 1-indexed)
    """
    config = FORMATTING_CONFIG.get(tab_key)
    if config is None:
        print(f"  ⚠️  apply_tab_formatting: '{tab_key}' sem config — formatação pulada.")
        return

    n_rows = len(df)
    if n_rows == 0:
        return

    if start_row < 3:
        raise ValueError(
            f"apply_tab_formatting: start_row={start_row} inseguro. "
            f"Formatação nunca deve tocar nas linhas 1 (banner) ou 2 (headers)."
        )

    # sheetId inteiro exigido pela Sheets API
    sheet_id = getattr(worksheet, "id", None)
    if sheet_id is None:
        sheet_id = worksheet._properties["sheetId"]
    n_cols   = config["n_cols"]

    reqs  = []
    reqs += build_base_requests(sheet_id, start_row, n_rows, n_cols)
    reqs += build_column_format_requests(sheet_id, start_row, n_rows, config)
    reqs += build_conditional_cell_requests(sheet_id, start_row, df, config)

    if reqs:
        spreadsheet.batch_update({"requests": reqs})
        print(f"    ↳ formatação: {len(reqs)} requests → '{worksheet.title}'")

print("✅ Funções de formatação Sheets API definidas")


In [ ]:
# ── Helpers de autenticação e escrita segura no Google Sheets ─────────────────
import re
import subprocess
import gspread
import google.oauth2.credentials
import google.oauth2.service_account
from gspread_dataframe import set_with_dataframe


def get_gspread_client() -> gspread.Client:
    "Retorna cliente gspread autenticado. Prioridade: 1) GOOGLE_SERVICE_ACCOUNT_JSON (Deepnote/CI), 2) gcloud token (local)."
    sa_json = os.getenv("GOOGLE_SERVICE_ACCOUNT_JSON")
    if sa_json:
        info = json.loads(sa_json)
        creds = google.oauth2.service_account.Credentials.from_service_account_info(
            info,
            scopes=[
                "https://www.googleapis.com/auth/spreadsheets",
                "https://www.googleapis.com/auth/drive",
            ],
        )
        print("✅ GSheets auth: service account (GOOGLE_SERVICE_ACCOUNT_JSON)")
        return gspread.authorize(creds)

    # Fallback local: gcloud user credentials
    result = subprocess.run(
        ["gcloud", "auth", "print-access-token"],
        capture_output=True, text=True, check=True,
    )
    access_token = result.stdout.strip()
    creds = google.oauth2.credentials.Credentials(token=access_token)
    print("✅ GSheets auth: gcloud user credentials (local fallback)")
    return gspread.Client(auth=creds)


# ── Configuração das abas da planilha existente ───────────────────────────────
TAB_CONFIG = {
    "farol_completo": {
        "worksheet_name": "🗂️ Farol Completo",
        "data_range_prefix": "A3:R",
        "header_range": "A2:R2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": "A1",
        "title_template": "🗂️  FAROL COMPLETO — {n} Produtos com Evidência Suficiente",
        "expected_headers": FAROL_COMPLETO_HEADERS,
    },
    "priorizar_melhoria": {
        "worksheet_name": "🔴 Priorizar Melhoria",
        "data_range_prefix": "A3:Q",
        "header_range": "A2:Q2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": "A1",
        "title_template": "🔴  PRIORIZAR MELHORIA — Ranking de Urgência   ·   {n} produtos identificados",
        "expected_headers": PRIORIZAR_MELHORIA_HEADERS,
    },
    "relatorio_pf": {
        "worksheet_name": "📋 Relatório PF",
        "data_range_prefix": "A3:R",
        "header_range": "A2:R2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": None,
        "title_template": None,
        "expected_headers": RELATORIO_PF_HEADERS,
    },
    "benchmark_categoria": {
        "worksheet_name": "📂 Benchmark Categoria",
        "data_range_prefix": "A3:J",
        "header_range": "A2:J2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": None,
        "title_template": None,
        "expected_headers": BENCHMARK_CATEGORIA_HEADERS,
    },
    "top_problemas": {
        "worksheet_name": "🏷️ Top Problemas",
        "data_range_prefix": "A3:F",
        "header_range": "A2:F2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": None,
        "title_template": None,
        "expected_headers": TOP_PROBLEMAS_HEADERS,
    },
    "lovable_priorizacao": {
        "worksheet_name": "lovable_priorizacao",
        "data_range_prefix": "A3:H",
        "header_range": "A2:H2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": "A1",
        "title_template": "🎯  PRIORIZAÇÃO LOVABLE v0 — Árvore de Decisão   ·   {n} produtos classificados",
        "expected_headers": LOVABLE_PRIORIZACAO_HEADERS,
    },
}


# ── Helpers seguros — nunca criam abas, nunca limpam aba inteira ──────────────
def open_target_spreadsheet(gc: gspread.Client, spreadsheet_id: str = SPREADSHEET_ID):
    spreadsheet = gc.open_by_key(spreadsheet_id)
    actual_id = getattr(spreadsheet, "id", spreadsheet_id)
    if actual_id != spreadsheet_id:
        raise RuntimeError(
            f"Planilha aberta não corresponde ao SPREADSHEET_ID esperado. "
            f"Esperado: {spreadsheet_id}. Atual: {actual_id}"
        )
    return spreadsheet


def get_required_worksheet(spreadsheet: gspread.Spreadsheet, worksheet_name: str):
    try:
        return spreadsheet.worksheet(worksheet_name)
    except Exception as exc:
        available = [ws.title for ws in spreadsheet.worksheets()]
        raise RuntimeError(
            f"Aba esperada não encontrada: '{worksheet_name}'. "
            f"Abas disponíveis: {available}. "
            f"Abortando para evitar criar ou desformatar abas."
        ) from exc


def assert_headers_match(worksheet, expected_headers: list, header_range: str) -> None:
    values = worksheet.get(header_range)
    if not values:
        raise ValueError(
            f"Header vazio no range {worksheet.title}!{header_range}. Abortando escrita."
        )
    actual_headers = [str(h).strip() for h in values[0]]
    expected_clean = [str(h).strip() for h in expected_headers]
    if actual_headers != expected_clean:
        raise ValueError(
            f"Headers incompatíveis na aba '{worksheet.title}'.\n"
            f"Esperado: {expected_clean}\n"
            f"Atual   : {actual_headers}\n"
            f"Abortando para evitar sobrescrever dados em layout incompatível."
        )


def _range_with_last_row(data_range_prefix: str, last_row: int) -> str:
    if not data_range_prefix[-1].isalpha():
        return data_range_prefix
    return f"{data_range_prefix}{last_row}"


def clear_data_body_only(worksheet, data_range_prefix: str, min_rows_to_clear: int = 1000) -> None:
    "Limpa somente o corpo da tabela. Nunca inclui linhas 1–2. Nunca usa worksheet.clear()."
    last_row = max(worksheet.row_count, min_rows_to_clear)
    closed_range = _range_with_last_row(data_range_prefix, last_row)
    if not re.match(r"^[A-Z]+3:", closed_range):
        raise ValueError(
            f"Range de limpeza inseguro: '{closed_range}'. Deve começar na linha 3."
        )
    worksheet.batch_clear([closed_range])


def ensure_enough_rows(worksheet, required_rows: int) -> None:
    "Adiciona linhas se necessário. Não redimensiona para baixo."
    if worksheet.row_count < required_rows:
        worksheet.add_rows(required_rows - worksheet.row_count)


def write_dataframe_body(worksheet, df: pd.DataFrame, start_row: int = 3, start_col: int = 1) -> None:
    "Escreve DataFrame sem header e sem index, preservando a formatação existente."
    if start_row < 3:
        raise ValueError(
            f"start_row inseguro: {start_row}. Dados devem começar na linha 3 ou abaixo."
        )
    required_rows = start_row + max(len(df), 1) + 20
    ensure_enough_rows(worksheet, required_rows)
    set_with_dataframe(
        worksheet,
        df.astype(object).where(pd.notna(df), ""),
        row=start_row,
        col=start_col,
        include_index=False,
        include_column_header=False,
        resize=False,
    )


def update_title_if_needed(worksheet, title_cell, title) -> None:
    if title_cell and title:
        worksheet.update([[title]], title_cell)


def update_tab_from_dataframe(
    spreadsheet: gspread.Spreadsheet,
    tab_key: str,
    df: pd.DataFrame,
    title_n: "int | None" = None,
) -> None:
    config = TAB_CONFIG[tab_key]
    worksheet = get_required_worksheet(spreadsheet, config["worksheet_name"])
    assert_headers_match(worksheet, config["expected_headers"], config["header_range"])
    if list(df.columns) != config["expected_headers"]:
        raise ValueError(
            f"Payload de '{tab_key}' não bate com headers esperados.\n"
            f"Esperado: {config['expected_headers']}\n"
            f"Atual   : {list(df.columns)}"
        )
    clear_data_body_only(worksheet, config["data_range_prefix"], max(len(df) + 20, 1000))
    write_dataframe_body(worksheet, df, config["start_row"], config["start_col"])
    # ── Reaplica formatação a todas as linhas de dados após escrita ────────
    apply_tab_formatting(
        spreadsheet,
        worksheet,
        df,
        tab_key,
        start_row=config["start_row"],
    )
    if config["title_template"] and config["title_cell"]:
        n = title_n if title_n is not None else len(df)
        update_title_if_needed(worksheet, config["title_cell"], config["title_template"].format(n=n))
    print(f"  ✓ {config['worksheet_name']}: {len(df):,} linhas × {len(df.columns)} colunas")


def update_resumo_executivo(spreadsheet: gspread.Spreadsheet, resumo_payload: dict) -> None:
    "Atualiza apenas células específicas da aba Resumo Executivo. Não limpa a aba."
    worksheet = get_required_worksheet(spreadsheet, "📊 Resumo Executivo")
    if "subtitle" in resumo_payload:
        worksheet.update([[resumo_payload["subtitle"]]], "A2")
    bucket_cards = resumo_payload.get("bucket_cards", {})
    # ⚠️ Validar cell_map visualmente na planilha antes do primeiro uso em produção.
    cell_map = {
        "Priorizar melhoria":          {"produtos": "B6", "receita_liq_fmt": "B7", "itens_retornados_fmt": "B8"},
        "Alerta em produto relevante": {"produtos": "D6", "receita_liq_fmt": "D7", "itens_retornados_fmt": "D8"},
        "Monitorar":                   {"produtos": "F6", "receita_liq_fmt": "F7", "itens_retornados_fmt": "F8"},
        "Não priorizar agora":         {"produtos": "H6", "receita_liq_fmt": "H7", "itens_retornados_fmt": "H8"},
        "Sem evidência suficiente":    {"produtos": "J6", "receita_liq_fmt": "J7", "itens_retornados_fmt": "J8"},
    }
    updates = []
    for bucket, fields in cell_map.items():
        values = bucket_cards.get(bucket, {})
        for field, cell in fields.items():
            updates.append({"range": cell, "values": [[values.get(field, 0)]]})
    if updates:
        worksheet.batch_update(updates)
    print("  ✓ 📊 Resumo Executivo atualizado")


def update_all_tabs(
    spreadsheet: gspread.Spreadsheet,
    farol_df: pd.DataFrame,
    priorizar_df: pd.DataFrame,
    relatorio_pf_df: "pd.DataFrame | None",
    benchmark_df: pd.DataFrame,
    top_problemas_df: pd.DataFrame,
    resumo_payload: dict,
    lovable_df: "pd.DataFrame | None" = None,
) -> None:
    """
    Atualiza a planilha existente preservando layout completo.

    Garantias:
    - NÃO cria planilha nova.
    - NÃO cria abas novas.
    - NÃO limpa abas inteiras (usa batch_clear somente no corpo, a partir de linha 3).
    - NÃO sobrescreve headers (linha 2) nem títulos mesclados (linha 1).
    - NÃO escreve index ou header no DataFrame.
    - NÃO altera a aba ℹ️ Legenda.
    - APLICA formatação completa (fundo alternado, bordas, cores condicionais)
      via Sheets API batchUpdate após cada write_dataframe_body.
    """
    update_tab_from_dataframe(spreadsheet, "priorizar_melhoria", priorizar_df, len(priorizar_df))
    if relatorio_pf_df is not None and not relatorio_pf_df.empty:
        update_tab_from_dataframe(spreadsheet, "relatorio_pf", relatorio_pf_df)
    else:
        print("  ⚠️  📋 Relatório PF: sem dados disponíveis — aba não atualizada.")
    update_tab_from_dataframe(spreadsheet, "benchmark_categoria", benchmark_df)
    update_tab_from_dataframe(spreadsheet, "top_problemas", top_problemas_df)
    update_tab_from_dataframe(spreadsheet, "farol_completo", farol_df, len(farol_df))
    if lovable_df is not None and not lovable_df.empty:
        update_tab_from_dataframe(spreadsheet, "lovable_priorizacao", lovable_df, len(lovable_df))
    else:
        print("  ⚠️  lovable_priorizacao: sem dados — aba não atualizada.")
    # Resumo por último: reflete sucesso completo das abas tabulares.
    update_resumo_executivo(spreadsheet, resumo_payload)


print("✅ Helpers de escrita segura no GSheets definidos")


In [ ]:
# ── QA pré-escrita ────────────────────────────────────────────────────────────
assert_output_headers(farol_completo_df, FAROL_COMPLETO_HEADERS, "farol_completo_df")
assert_output_headers(priorizar_melhoria_df, PRIORIZAR_MELHORIA_HEADERS, "priorizar_melhoria_df")
assert_output_headers(benchmark_categoria_df, BENCHMARK_CATEGORIA_HEADERS, "benchmark_categoria_df")
assert_output_headers(top_problemas_df, TOP_PROBLEMAS_HEADERS, "top_problemas_df")
assert_output_headers(lovable_priorizacao_df, LOVABLE_PRIORIZACAO_HEADERS, "lovable_priorizacao_df")
if relatorio_pf_df is not None:
    assert_output_headers(relatorio_pf_df, RELATORIO_PF_HEADERS, "relatorio_pf_df")

if farol_completo_df.empty:
    raise ValueError("farol_completo_df vazio. Abortando escrita.")

if priorizar_melhoria_df.empty:
    warnings.warn("priorizar_melhoria_df vazio. Aba Priorizar Melhoria será atualizada sem linhas de dados.")

if relatorio_pf_df is None or relatorio_pf_df.empty:
    warnings.warn("relatorio_pf_df vazio/None. Aba Relatório PF não será atualizada.")

if lovable_priorizacao_df.empty:
    warnings.warn("lovable_priorizacao_df vazio. Aba lovable_priorizacao não será atualizada.")

# ── Execução: atualiza Google Sheets ou reporta skip ──────────────────────────
if UPDATE_GOOGLE_SHEETS:
    gc = get_gspread_client()
    spreadsheet = open_target_spreadsheet(gc, SPREADSHEET_ID)
    print(f"Planilha aberta: {spreadsheet.title}")
    print(f"URL: https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")

    update_all_tabs(
        spreadsheet=spreadsheet,
        farol_df=farol_completo_df,
        priorizar_df=priorizar_melhoria_df,
        relatorio_pf_df=relatorio_pf_df,
        benchmark_df=benchmark_categoria_df,
        top_problemas_df=top_problemas_df,
        resumo_payload=resumo_payload,
        lovable_df=lovable_priorizacao_df,
    )

    print("\n✅ Google Sheets atualizado com sucesso, sem recriar planilha ou limpar layout")
    print(f"   https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")
else:
    print("⏭️  UPDATE_GOOGLE_SHEETS=False — planilha não atualizada.")

# ── Export debug Excel (opcional, não é output produtivo) ─────────────────────
if EXPORT_DEBUG_FILES:
    debug_path = OUTPUT_DIR / f"td_priorizacao_debug_{DATE_TAG}.xlsx"
    export_priority_workbook(
        executive_df,
        str(debug_path),
        scorecard_view_df=scorecard_view_df,

    )

    print(f"\n✅ Excel executivo de debug gerado: {debug_path}")    print("⏭️  EXPORT_DEBUG_FILES=False — arquivo Excel não gerado.")
else:

## 12. Log final de execução

Resumo de todas as métricas do pipeline após execução completa.

**Entrada:** todos os DataFrames produzidos
**Saída:** log no stdout
**Quando mexer:** ao adicionar novo DataFrame ao pipeline.


In [ ]:
print("=" * 70)
print("✅ PIPELINE TD — PRIORIZAÇÃO DE MELHORIAS PRODUTO FÍSICO")
print("=" * 70)
print(f"  Run date               : {RUN_DATE}")
print(f"  Date tag               : {DATE_TAG}")
print()
print(f"  executive_df           : {executive_df.shape[0]:,} produtos")
print(f"  executive_prepared_df  : {executive_prepared_df.shape[0]:,} linhas")
print(f"  farol_completo_df      : {farol_completo_df.shape}")
print(f"  priorizar_melhoria_df  : {priorizar_melhoria_df.shape}")
print(f"  relatorio_pf_df        : {relatorio_pf_df.shape if relatorio_pf_df is not None else 'None'}")
print(f"  benchmark_categoria_df : {benchmark_categoria_df.shape}")
print(f"  top_problemas_df       : {top_problemas_df.shape}")
print(f"  lovable_priorizacao_df : {lovable_priorizacao_df.shape}")
print()
print("── Distribuição lovable_priorizacao ────────────────────────────────────")
print(lovable_priorizacao_df["Priorização"].value_counts().to_string())
print(f"  Thresholds: taxa_devol > {LOVABLE_THRESHOLD_TAXA_DEVOLUCAO:.0%}, tag_conc > {LOVABLE_THRESHOLD_TAG_CONCENTRACAO:.0%}")
print()
print(f"  EXPORT_DEBUG_FILES     : {EXPORT_DEBUG_FILES}")
print(f"  UPDATE_GOOGLE_SHEETS   : {UPDATE_GOOGLE_SHEETS}")
if UPDATE_GOOGLE_SHEETS:
    print(f"  Planilha               : https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")
print("=" * 70)